# FinSight Lanka: Customer Data Analysis: Workings
**Dialog Finance PLC | Analytics Trainee Assessment**  |  Prepared by: *Amanda Sewwandi*


### Map of this notebook
| Section | Content |
|---|---|
| **0. Data audit and cleaning** | Every data-quality issue found, how it was handled, and why (0.1 - 0.12). One cleaning log; one validation step. |
| **Q1. Savings and customer behaviour** | Q1A average savings by segment / province / urban-rural · Q1B age group with highest total savings · Q1C growing vs draining |
| **Q2. Loan portfolio and risk** | Q2A loan penetration · Q2B debt-to-savings ratio · Q2C default patterns · Q2D sensitivity analysis |
| **Q3. Digital engagement and product holdings** | Q3A app users vs savings · Q3B cross-sell opportunity · Q3C best acquisition channel |
| **Q4. Five KPIs and dashboard** | KPI definitions, formulas, current values, targets · dashboard (PDF export) |
| **Q5. Board summary and recommendations** | Findings, two recommendations, two additional data sources (every figure re-computed from the data) |
| **Q6. Critique of my own work** | Eight criticisms with severity, impact and remedy (with the tests behind them) |

### Conventions used throughout
1. **Flag, don't silently delete.** An impossible *value* becomes missing; the *customer* stays for unrelated analyses.
2. **Never invent a number.** Missing balances are not imputed.
3. **Every cleaning decision is logged** (what, how many rows, why).
4. **Median next to mean, uncertainty next to rates.** Balances are skewed and several samples are small, so confidence intervals and tests accompany most results.
5. All judgement calls are stated as assumptions and re-tested where they matter (see Q2D and Q6).

### How to run
Place the Excel file `20260512_FintechTrainee_CaseStudy_Dataset_FinSightLanka_F.xlsx` in the same folder as this notebook, then **Run All**. Requires Python 3.10+, `pandas>=2.2`, `numpy`, `scipy`, `matplotlib>=3.9`, `openpyxl`. Outputs (charts, Excel tables, the dashboard PDF) are written to `workings_outputs/`.

## 0. Data audit and cleaning
### 0.1 Setup and configuration

In [ ]:
import sys
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import stats
from pathlib import Path
from IPython.display import display

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def _v(s): return tuple(int(x) for x in s.split(".")[:2])
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")
if _v(pd.__version__) < (2, 2): print("WARNING: pandas >= 2.2 is required")
if _v(matplotlib.__version__) < (3, 9): print("WARNING: matplotlib >= 3.9 is required")

FILE_NAME = "20260512_FintechTrainee_CaseStudy_Dataset_FinSightLanka_F.xlsx"
DATA_PATH = Path(FILE_NAME) if Path(FILE_NAME).exists() else Path("/mnt/user-data/uploads") / FILE_NAME
SHEET_NAME = "Customer_Data"

AS_OF            = pd.Timestamp("2024-01-01")   
AGE_MIN, AGE_MAX = 18, 90                       
SAVINGS_MAX      = 100_000_000                  
WITHDRAWAL_MAX   = 1_000_000                    
ELEVATED_MULT, ELEVATED_MIN_DEFAULTS = 1.5, 3   
N_PERM, N_BOOT   = 5000, 3000                   

OUT_ROOT = Path("workings_outputs"); OUT_ROOT.mkdir(exist_ok=True)
OUT = OUT_ROOT
rng = np.random.default_rng(42)
cleaning_log = []
def log(step, issue, rows, action, why):
    cleaning_log.append({"Step": step, "Issue": issue, "Rows affected": int(rows), "Action": action, "Rationale": why})
    print(f"[{step}] {issue}: {int(rows)} row(s) -> {action}")

def lkr_fmt(x, _=None):
    return f"{x/1e6:.1f}M" if abs(x) >= 1e6 else f"{x/1e3:.0f}k"

### 0.2 Load the raw data and audit it


In [ ]:
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = raw.copy()                       
print("Raw shape:", raw.shape, "| unique Customer_IDs:", raw["Customer_ID"].nunique())

miss = raw.isna().sum(); miss = miss[miss > 0].to_frame("missing").assign(pct=lambda x: x["missing"] / len(raw) * 100)
print("\nMissing values (loan-detail columns are structurally blank for non-borrowers: not an error):")
display(miss)

audit_cols = ["Age", "Savings_Balance", "Monthly_Deposit_Avg", "Monthly_Withdrawal_Avg", "Net_Monthly_Flow"]
display(raw[audit_cols].describe().T[["count", "min", "50%", "max"]])
print("Top 5 savings values:", raw["Savings_Balance"].nlargest(5).round(0).tolist())
print("Top 3 withdrawal values:", raw["Monthly_Withdrawal_Avg"].nlargest(3).round(0).tolist())
print("Ages outside 18-90:", raw.loc[~raw["Age"].between(AGE_MIN, AGE_MAX), "Age"].tolist())
print("Exact duplicate rows:", int(raw.duplicated().sum()), "| duplicate Customer_IDs:", int(raw["Customer_ID"].duplicated().sum()))
for c in ["Gender", "Account_Status", "District"]:
    print(f"{c} values:", sorted(raw[c].dropna().unique().tolist()))

### 0.3 Standardise text, districts and provinces
Typos and inconsistent case would split one group into several in any `groupby`.

In [ ]:
for c in [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object]:
    df[c] = df[c].str.strip()

b = df["Gender"].copy(); df["Gender"] = df["Gender"].str.upper().str[0].map({"M": "Male", "F": "Female"})
log("0.3a", "Gender written inconsistently (M, male, FEMALE, f...)", (b != df["Gender"]).sum(), "Mapped to Male/Female", "Consistent categories")
b = df["Account_Status"].copy(); df["Account_Status"] = df["Account_Status"].str.title()
log("0.3b", "Account_Status inconsistent case/spaces", (b != df["Account_Status"]).sum(), "Title-cased", "Consistent categories")
b = df["District"].copy()
df["District"] = df["District"].replace({"Columbo": "Colombo", "Kanady": "Kandy", "Gale": "Galle", "Jafna": "Jaffna"})
log("0.3c", "District misspellings (Columbo, Kanady, Gale, Jafna)", (b != df["District"]).sum(), "Corrected", "Typos would create fake districts")

d2p = {"Colombo": "Western", "Gampaha": "Western", "Kandy": "Central", "Galle": "Southern", "Matara": "Southern",
       "Jaffna": "Northern", "Kurunegala": "North Western", "Ratnapura": "Sabaragamuwa", "Trincomalee": "Eastern", "Badulla": "Uva"}
b = df["Province"].copy(); df["Province"] = df["District"].map(d2p).fillna(df["Province"])
log("0.3d", "Province contradicts District (rows with a misspelt district carried the wrong province)", (b != df["Province"]).sum(),
    "Province rebuilt from District lookup", "District is the more granular field, so it is the source of truth")
display(pd.crosstab(df["District"], df["Province"]))

### 0.4 Duplicates
Two kinds, handled differently:
- **Exact copies** → dropped.
- **Same Customer_ID with conflicting values** → *resolved by evidence*: keep the copy with a plausible age and a NIC birth-year that agrees with the date of birth (old-format NICs begin with the birth year). Quarantining all copies is also defensible, but it would discard 5 genuine customers where the conflicts are trivial (a KYC status, a login date, a NIC typo). The effect of this choice is tested in Q6.

In [ ]:
def calc_age(dob, as_of=AS_OF):
    had_bday = (as_of.month > dob.dt.month) | ((as_of.month == dob.dt.month) & (as_of.day >= dob.dt.day))
    return (as_of.year - dob.dt.year) - (~had_bday).astype(int)

df["Age_Calc"] = calc_age(df["Date_of_Birth"])
nic_year = 1900 + pd.to_numeric(df["NIC_Number"].str[:2], errors="coerce")
df["NIC_DOB_OK"] = np.where(df["Date_of_Birth"].dt.year < 2000, nic_year == df["Date_of_Birth"].dt.year, True)

conflict_view = df[df["Customer_ID"].duplicated(keep=False) & ~df.duplicated(keep=False)].sort_values("Customer_ID")
print("Same-ID rows that differ in some column (before resolving):")
display(conflict_view[["Customer_ID", "KYC_Status", "NIC_Number", "Date_of_Birth", "Age", "Age_Calc", "NIC_DOB_OK", "Savings_Balance"]])

exact = df.duplicated()
df = df[~exact]
log("0.4a", "Exact duplicate rows (identical in every column)", exact.sum(), "Dropped the copies", "Same record recorded twice")

df["_score"] = df["Age_Calc"].between(AGE_MIN, AGE_MAX).astype(int) + df["NIC_DOB_OK"].astype(int)
n0 = len(df)
df = (df.sort_values(["Customer_ID", "_score"], ascending=[True, False], kind="mergesort")
        .drop_duplicates("Customer_ID").sort_index().drop(columns="_score"))
log("0.4b", "Same Customer_ID with conflicting values", n0 - len(df), "Kept the copy that passes age/NIC validation", "Resolved by evidence, not by position in the file")
assert df["Customer_ID"].is_unique
CONFLICT_IDS = conflict_view["Customer_ID"].unique().tolist()      # kept for the sensitivity test in Q6

df["Flag_NIC_Shared"] = df["NIC_Number"].duplicated(keep=False)
log("0.4c", "Same NIC used by different Customer_IDs", df["Flag_NIC_Shared"].sum(), "Flagged only (rows kept)", "Possible identity/KYC issue; not enough evidence to delete")
df["Flag_NIC_DOB_Mismatch"] = ~df["NIC_DOB_OK"]
log("0.4d", "NIC birth-year disagrees with date of birth (born pre-2000)", df["Flag_NIC_DOB_Mismatch"].sum(), "Flagged only", "Informational: not used in any metric")
print("Customers after de-duplication:", len(df))

### 0.5 Age: recompute from Date of Birth
The reported `Age` is unreliable. Some rows are off by exactly 1 year (January birthdays: the column was evidently calculated around mid-January 2024), and others are plainly wrong (DOB 1974 with Age = 99; DOB 1975 with Age = 150). Date of birth is a harder fact than a typed number, so age is **recomputed at 1 Jan 2024**. Anything still outside 18-90 is genuinely impossible (for example a customer born in 2020 holding a loan).

In [ ]:
age_gap = (df["Age"] - df["Age_Calc"]).abs()
log("0.5a", "Reported Age off by exactly 1 year vs DOB (January birthdays)", (age_gap == 1).sum(), "Recomputed from DOB", "Benign reference-date effect")
log("0.5b", "Reported Age off by more than 1 year vs DOB (e.g. 99, 150)", (age_gap > 1).sum(), "Recomputed from DOB", "Would have created a false '65+' customer")
bad_age = ~df["Age_Calc"].between(AGE_MIN, AGE_MAX)
df["Flag_Age_Invalid"] = bad_age
df["Age"] = df["Age_Calc"].where(~bad_age)
log("0.5c", f"Age outside {AGE_MIN}-{AGE_MAX} after recompute", bad_age.sum(), "Age set to missing + flagged", "Excluded from age-band analysis only; customer kept elsewhere")
df = df.drop(columns=["Age_Calc", "NIC_DOB_OK"])

### 0.6 Impossible balances and flows
- **Savings_Balance:** negative balances and a 999,999,999 placeholder are set to **missing** (never imputed).
- **Monthly_Withdrawal_Avg:** two placeholder values (9,999,999 / 8,888,888). `Net_Monthly_Flow = Deposit − Withdrawal` holds *exactly* on every other row, so for these two the **net-flow field is the trustworthy one** and the withdrawal is back-solved from it. This is better than discarding the rows, and it prevents two false "draining" customers.

In [ ]:
bad_sav = (df["Savings_Balance"] < 0) | (df["Savings_Balance"] > SAVINGS_MAX)
display(df.loc[bad_sav, ["Customer_ID", "Customer_Segment", "Province", "Urban_Rural", "Savings_Balance"]])
df["Flag_Savings_Invalid"] = bad_sav
df.loc[bad_sav, "Savings_Balance"] = np.nan
log("0.6a", "Impossible savings (negative or >LKR 100m placeholder)", bad_sav.sum(), "Set to missing + flagged", "A made-up balance would distort averages and totals")

ok = df["Monthly_Withdrawal_Avg"] <= WITHDRAWAL_MAX
gap_ok = (df.loc[ok, "Monthly_Deposit_Avg"] - df.loc[ok, "Monthly_Withdrawal_Avg"] - df.loc[ok, "Net_Monthly_Flow"]).abs().dropna()
print(f"Identity Deposit - Withdrawal = Net flow breaks on {(gap_ok > 1).sum()} of {len(gap_ok)} valid rows")
bad_wd = df["Monthly_Withdrawal_Avg"] > WITHDRAWAL_MAX
display(df.loc[bad_wd, ["Customer_ID", "Monthly_Deposit_Avg", "Monthly_Withdrawal_Avg", "Net_Monthly_Flow"]])
df["Flag_Withdrawal_Reconstructed"] = bad_wd
df.loc[bad_wd, "Monthly_Withdrawal_Avg"] = df["Monthly_Deposit_Avg"] - df["Net_Monthly_Flow"]
log("0.6b", "Placeholder withdrawals (9,999,999 / 8,888,888)", bad_wd.sum(), "Reconstructed = Deposit - Net_Monthly_Flow", "Identity holds on every other row")

### 0.7 Dates
Impossible dates (after the snapshot) become missing. Conflicts that could have an innocent explanation are **flagged, not changed**.

In [ ]:
for col in ["Account_Open_Date", "Last_Login_Date", "Loan_Start_Date", "KYC_Last_Updated"]:
    fut = df[col] > AS_OF
    df.loc[fut, col] = pd.NaT
    log("0.7a", f"{col} after the snapshot date ({AS_OF.date()})", fut.sum(), "Set to missing" if fut.any() else "None found", "A future date cannot be real")

lb_open = df["Loan_Start_Date"] < df["Account_Open_Date"]
lb_18 = df["Loan_Start_Date"] < (df["Date_of_Birth"] + pd.DateOffset(years=18))
df["Flag_Loan_Date_Conflict"] = (lb_open | lb_18).fillna(False)
log("0.7b", "Loan started before account opened or before age 18", df["Flag_Loan_Date_Conflict"].sum(), "Flagged only", "Too few to delete; used as a sensitivity test in Q2D")
df["Flag_KYC_Before_Open"] = (df["KYC_Last_Updated"] < df["Account_Open_Date"]).fillna(False)
log("0.7c", "KYC_Last_Updated earlier than Account_Open_Date", df["Flag_KYC_Before_Open"].sum(), "Noted only", "Too widespread to be isolated typos; KYC dates are not used in any metric")

### 0.8 Missing Customer_Segment
Segment is *largely* explained by interest-rate tier plus balance, but **not perfectly** (Premium and Regular overlap in balance). A simple rule is built and **back-tested on the labelled rows** before it is trusted on the five unlabelled ones.

In [ ]:
lab = df[df["Customer_Segment"].notna()]
display(lab.groupby("Customer_Segment")["Savings_Balance"].agg(["min", "max", "count"]))
starter_max = lab.loc[lab["Customer_Segment"] == "Starter", "Savings_Balance"].max()

def infer_segment(r):
    if r["Interest_Rate_Tier"] == "Gold":   return "Premium"
    if r["Interest_Rate_Tier"] == "Silver": return "Regular"
    if pd.isna(r["Savings_Balance"]):       return np.nan
    return "Starter" if r["Savings_Balance"] <= starter_max else "Regular"

pred = lab.apply(infer_segment, axis=1)
acc = (pred == lab["Customer_Segment"]).mean()
print(f"Rule accuracy on {len(lab)} labelled rows: {acc:.1%} (weak spot: Standard-tier customers with small balances cannot be reliably split into Starter vs Regular)")
display(pd.crosstab(lab["Customer_Segment"], pred, rownames=["Actual"], colnames=["Predicted"]))

miss_seg = df["Customer_Segment"].isna()
df["Flag_Segment_Imputed"] = miss_seg
df.loc[miss_seg, "Customer_Segment"] = df.loc[miss_seg].apply(infer_segment, axis=1)
log("0.8", "Customer_Segment missing", miss_seg.sum(), f"Inferred from tier + balance (back-test accuracy {acc:.0%})", "Better than dropping 5 customers; flagged so results can be re-run without them")

### 0.9 Missing KYC status
For a compliance measure, "unknown" is itself a risk and must stay visible, so blanks are **labelled "Unknown"** rather than guessed or dropped.

In [ ]:
miss_kyc = df["KYC_Status"].isna()
df["KYC_Status"] = df["KYC_Status"].fillna("Unknown")
log("0.9", "KYC_Status missing", miss_kyc.sum(), "Labelled 'Unknown'", "Unverified status is a compliance risk: keep it visible")

### 0.10 The loan-holder rule (the most important cleaning decision for Q2-Q4)
`Has_Loan` contradicts the loan-detail columns in a handful of rows. **Rule:**
- **Loan_Holder** = `Has_Loan == "Yes"` **or** any loan detail is present (type, outstanding balance or repayment status). Details are stronger evidence than a flag.
- **Loan_Data_Complete** = Loan_Type **and** Outstanding_Loan_Balance both present. Only these loans enter the debt-to-savings and default-rate calculations.
- `Has_Loan` blank with *no* loan details → not a holder, flagged as unknown.

Q2D re-runs the headline results under alternative rules.

In [ ]:
has_details = df["Loan_Type"].notna() | df["Outstanding_Loan_Balance"].notna() | df["Loan_Repayment_Status"].notna()
df["Loan_Holder"] = (df["Has_Loan"] == "Yes") | has_details
df["Loan_Data_Complete"] = df["Loan_Type"].notna() & df["Outstanding_Loan_Balance"].notna()
df["Loan_Status_Unknown"] = df["Has_Loan"].isna() & ~has_details

log("0.10a", "Has_Loan = No but loan details exist", ((df["Has_Loan"] == "No") & has_details).sum(), "Treated as loan holder", "Details outweigh a single flag")
log("0.10b", "Has_Loan blank but loan details exist", (df["Has_Loan"].isna() & has_details).sum(), "Treated as loan holder", "Details outweigh a missing flag")
log("0.10c", "Loan holders with incomplete data (excluded from DSR / default rates)", (df["Loan_Holder"] & ~df["Loan_Data_Complete"]).sum(), "Excluded from DSR/default calculations", "Cannot compute exposure or classify type")
log("0.10d", "Has_Loan blank and no loan details", df["Loan_Status_Unknown"].sum(), "Non-holder + Loan_Status_Unknown flag", "No evidence either way")
print(f"Loan holders: {df['Loan_Holder'].sum()} | complete data: {df['Loan_Data_Complete'].sum()} | unknown status: {df['Loan_Status_Unknown'].sum()}")
display(df.loc[df["Loan_Holder"] & ~df["Loan_Data_Complete"], ["Customer_ID", "Has_Loan", "Loan_Type", "Outstanding_Loan_Balance", "Loan_Repayment_Status"]])

### 0.11 Cross-column contradictions (flagged, not "fixed")
These are logically inconsistent, but there is no way to know which column is right. They are **flagged** so later analysis can test sensitivity, and they feed the limitations in Q6.

In [ ]:
df["Flag_App_Channel_Contradiction"] = (df["Preferred_Channel"] == "App") & (df["Mobile_App_User"] == "No")
df["Flag_Push_Without_App"] = (df["Push_Notification_Enabled"] == "Yes") & (df["Mobile_App_User"] == "No")
implied = (1 + (df["Has_Fixed_Deposit"] == "Yes").astype(int) + (df["Has_Insurance"] == "Yes").astype(int)
             + (df["Has_Mobile_Wallet"] == "Yes").astype(int) + df["Loan_Holder"].astype(int))
df["Flag_Products_Mismatch"] = df["Number_of_Products"] != implied
df["Flag_FD_gt_Savings"] = df["FD_Amount"] > df["Savings_Balance"]          # FDs appear to sit OUTSIDE the savings balance
for col, txt in [("Flag_App_Channel_Contradiction", "Preferred channel = App but not an app user"),
                 ("Flag_Push_Without_App", "Push notifications enabled but not an app user"),
                 ("Flag_Products_Mismatch", "Number_of_Products differs from the count implied by product flags"),
                 ("Flag_FD_gt_Savings", "FD_Amount larger than Savings_Balance")]:
    log("0.11", txt, df[col].sum(), "Flagged only", "Cannot tell which field is right; relevant to Q3 and Q6")

### 0.12 Validation (hard stops) and cleaning log

In [ ]:
assert df["Customer_ID"].is_unique and len(df) == 500
assert df["Savings_Balance"].dropna().between(0, SAVINGS_MAX).all()            # catches placeholders AND negatives
assert df["Monthly_Withdrawal_Avg"].le(WITHDRAWAL_MAX).all()
assert df["Age"].dropna().between(AGE_MIN, AGE_MAX).all()
assert df["Customer_Segment"].notna().all()
assert df["Gender"].isin(["Male", "Female"]).all()
assert df["Account_Status"].isin(["Active", "Inactive", "Suspended"]).all()
assert df["KYC_Status"].isin(["Verified", "Expired", "Pending", "Unknown"]).all()
chk = (df["Monthly_Deposit_Avg"] - df["Monthly_Withdrawal_Avg"] - df["Net_Monthly_Flow"]).abs().dropna()
assert (chk < 1).all()                                                          # flow identity holds everywhere
assert (df[["Account_Open_Date", "Last_Login_Date", "Loan_Start_Date", "KYC_Last_Updated"]].max() <= AS_OF).all()
_L = df[df["Loan_Data_Complete"]]
assert _L["Loan_Repayment_Status"].isin(["On-Time", "Delayed", "Defaulted"]).all()
assert _L["Loan_Type"].isin(["Business", "Housing", "Personal", "Vehicle"]).all()
_both = _L["Loan_Amount"].notna()
assert (_L.loc[_both, "Outstanding_Loan_Balance"] <= _L.loc[_both, "Loan_Amount"]).all()   # cannot owe more than borrowed
assert ((df["Monthly_Withdrawal_Avg"] / df["Monthly_Deposit_Avg"]).dropna() < 1).all()
print("All validation checks passed |", len(df), "customers |", len(_L), "loans with complete data")

summary = pd.DataFrame({
    "Raw": [len(raw), raw["Customer_ID"].nunique(), raw["Savings_Balance"].isna().sum(), raw["Savings_Balance"].mean()],
    "Cleaned": [len(df), df["Customer_ID"].nunique(), df["Savings_Balance"].isna().sum(), df["Savings_Balance"].mean()],
}, index=["Rows", "Unique customers", "Missing savings", "Mean savings (LKR)"])
display(summary)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].boxplot(raw["Savings_Balance"].dropna(), vert=False); ax[0].set_title("Savings, RAW (one placeholder squashes everything)")
ax[1].boxplot(df["Savings_Balance"].dropna(), vert=False); ax[1].set_title("Savings, CLEANED")
for a_ in ax: a_.set_xlabel("LKR")
plt.tight_layout(); plt.show()

CLEANING_LOG = pd.DataFrame(cleaning_log)
display(CLEANING_LOG[["Step", "Issue", "Rows affected", "Action"]])

---
# Q1. Savings and customer behaviour
Q1A average savings by Customer Segment, Province and Urban/Rural · Q1B age group with the highest total savings · Q1C customers growing vs draining their savings.

**Method:** mean *and* median (balances are right-skewed), valid balances only, sample sizes shown, and statistical tests so that "meaningful" means more than "visibly different".

In [ ]:
OUT = OUT_ROOT / "q1"; OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

---
# Q1A: Average savings by Segment, Province and Urban/Rural
**Method:** mean *and* median (balances are right-skewed, so the mean alone misleads), valid balances only, sample size shown, plus a **Kruskal-Wallis test** (does *any* group differ?, no normality assumption needed). A difference only counts as "meaningful" if it is both visible *and* not explainable by chance.

In [ ]:
def savings_table(col):
    g = df.groupby(col, dropna=False)["Savings_Balance"]
    t = g.agg(customers="size", valid_balances="count", mean="mean", median="median", total="sum")
    t["share_of_total_%"] = t["total"] / t["total"].sum() * 100
    return t.sort_values("mean", ascending=False)

def kruskal_p(col):
    groups = [x.dropna().values for _, x in df.groupby(col)["Savings_Balance"]]
    return stats.kruskal(*groups).pvalue

def mean_median_chart(t, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=False)
    for a, m, colr in zip(ax, ["mean", "median"], ["#1f4e79", "#2e8b57"]):
        bars = a.bar(t.index.astype(str), t[m], color=colr)
        a.set_title(f"{m.title()} savings: {title}"); a.yaxis.set_major_formatter(FuncFormatter(lkr_fmt))
        a.tick_params(axis="x", rotation=30)
        for bar, v in zip(bars, t[m]): a.text(bar.get_x() + bar.get_width()/2, v, lkr_fmt(v), ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / fname, dpi=150); plt.show()

q1a = {}
for col, f in [("Customer_Segment", "q1a_segment.png"), ("Province", "q1a_province.png"), ("Urban_Rural", "q1a_urban_rural.png")]:
    q1a[col] = savings_table(col)
    print(f"\n=== {col} | Kruskal-Wallis p = {kruskal_p(col):.4f} ===")
    display(q1a[col].round({"mean": 0, "median": 0, "total": 0, "share_of_total_%": 1}))
    mean_median_chart(q1a[col], col.replace("_", " "), f)

In [ ]:
# Pairwise follow-up for the only geographic split people will ask about: Urban vs Rural
u = df.loc[df["Urban_Rural"] == "Urban", "Savings_Balance"].dropna()
r = df.loc[df["Urban_Rural"] == "Rural", "Savings_Balance"].dropna()
p_ur = stats.mannwhitneyu(u, r).pvalue
print(f"Urban mean {u.mean():,.0f} vs Rural mean {r.mean():,.0f}  ({u.mean()/r.mean()-1:+.0%})")
print(f"Urban median {u.median():,.0f} vs Rural median {r.median():,.0f}  ({u.median()/r.median()-1:+.0%})")
print(f"Mann-Whitney p = {p_ur:.3f} ->", "significant" if p_ur < 0.05 else "NOT statistically significant")
sub = df.loc[df["Urban_Rural"] == "Suburban", "Savings_Balance"].dropna()
p_us = stats.mannwhitneyu(u, sub).pvalue
print(f"Urban vs Suburban (the lowest group): median {u.median():,.0f} vs {sub.median():,.0f}, Mann-Whitney p = {p_us:.3f}")
print(f"Caution: 3 pairwise comparisons were possible, so a Bonferroni-adjusted p is about {min(p_us*3, 1):.2f}: treat this as a hint, not a proven gap.")

# Sensitivity: does the story change if only ACTIVE accounts are used?
act = df[df["Account_Status"] == "Active"]
print("\nSensitivity (Active accounts only), median savings:")
for col in ["Province", "Urban_Rural"]:
    display(act.groupby(col)["Savings_Balance"].median().round(0).to_frame("median_active_only").T)

In [ ]:
# Draft wording generated from the numbers (so the text can never go stale)
seg, prov, ur = q1a["Customer_Segment"], q1a["Province"], q1a["Urban_Rural"]
p_prov, p_urr = kruskal_p("Province"), kruskal_p("Urban_Rural")
print("SEGMENT :", f"{seg.index[0]} holds the most per customer (mean {seg['mean'].iloc[0]:,.0f}) vs {seg.index[-1]} ({seg['mean'].iloc[-1]:,.0f}), "
      f"a {seg['mean'].iloc[0]/seg['mean'].iloc[-1]:.0f}x gap. Premium customers are {seg.loc['Premium','customers']/seg['customers'].sum():.0%} of customers but "
      f"{seg.loc['Premium','share_of_total_%']:.0f}% of all savings.")
print("          Caution: segments are assigned largely by balance/tier, so this gap is mostly by design, not a discovery.")
print("PROVINCE:", f"means range {prov['mean'].min():,.0f} ({prov['mean'].idxmin()}) to {prov['mean'].max():,.0f} ({prov['mean'].idxmax()}); "
      f"medians only {prov['median'].min():,.0f} to {prov['median'].max():,.0f}. Kruskal p = {p_prov:.2f} ->",
      "real difference" if p_prov < 0.05 else "no statistically meaningful provincial difference (gaps are driven by a few large balances).")
print("URBAN/RURAL:", f"Urban mean is {ur.loc['Urban','mean']/ur.loc['Rural','mean']-1:+.0%} vs Rural (Urban-vs-Rural p = {p_ur:.2f}, not conclusive), "
      f"but Urban median is {u.median()/sub.median()-1:+.0%} above Suburban (p = {p_us:.3f}; ~{min(p_us*3,1):.2f} after adjusting for 3 comparisons). "
      "Verdict: a weak urban advantage over suburban customers; rural is not distinguishable. Not a strong enough gap to design strategy around.")

---
# Q1B: Which age group holds the highest total savings?

**Banding logic (own definition):** 18-24 (early career / students), 25-34 (career building), 35-44 (peak family & mortgage years), 45-54 (peak earning, pre-retirement), 55-64 (pre-retirement), 65+ (retired). Lower bound 18 because customers must be adults; the 10-year width from 25 gives enough customers per band to compare.

*Total* savings = **how many customers** × **how much each holds**, so the table shows both, plus the share of the total, to separate "big group" from "rich group".

In [ ]:
bins   = [18, 25, 35, 45, 55, 65, np.inf]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["Age_Band"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

age = (df.groupby("Age_Band", observed=False)["Savings_Balance"]   # observed=False keeps EMPTY bands visible
         .agg(customers="size", valid_balances="count", total="sum", mean="mean", median="median"))
age["share_of_total_%"] = age["total"] / age["total"].sum() * 100
display(age.round({"total": 0, "mean": 0, "median": 0, "share_of_total_%": 1}))

top = age["total"].idxmax()
print(f"\nHighest TOTAL savings: {top}  (LKR {age.loc[top,'total']:,.0f} = {age.loc[top,'share_of_total_%']:.1f}% of all savings)")
print(f"Highest AVERAGE balance: {age['mean'].idxmax()}  | Highest MEDIAN: {age['median'].idxmax()}")
print("Customers aged 55+ in the data:", int((df['Age'] >= 55).sum()), " <- the customer base is under 55 (relevant for Q6)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(age.index.astype(str), age["total"], color=["#1f4e79" if i == top else "#9db7d1" for i in age.index])
ax[0].set_title("Total savings by age band"); ax[0].yaxis.set_major_formatter(FuncFormatter(lkr_fmt))
ax[1].bar(age.index.astype(str), age["customers"], color="#2e8b57"); ax[1].set_title("Number of customers by age band")
plt.tight_layout(); plt.savefig(OUT / "q1b_age_bands.png", dpi=150); plt.show()

In [ ]:
# Robustness: does the answer depend on how I drew the bands?
# Only compare schemes whose bands are (about) the same width, or equal-sized groups; a 20-year band
# will always beat a 10-year band, which proves nothing.
schemes = {
    "Main: 10-yr bands from 25":            dict(bins=[18, 25, 35, 45, 55, 65, np.inf]),
    "Shifted 10-yr bands (18-27, 28-37...)": dict(bins=[18, 28, 38, 48, 58, np.inf]),
    "5-yr bands (18-22, 23-27...)":         dict(bins=list(range(18, 63, 5)) + [np.inf]),
    "Equal customers (5 groups)":           dict(q=5),
}
rows = []
for name, kw in schemes.items():
    cut = pd.qcut(df["Age"], kw["q"]) if "q" in kw else pd.cut(df["Age"], bins=kw["bins"], right=False)
    g = df.groupby(cut, observed=True)["Savings_Balance"]
    tot = g.sum(); best = tot.idxmax()
    rows.append({"Banding": name, "Top band by total savings": str(best),
                 "Customers in it": int(df.groupby(cut, observed=True).size()[best]),
                 "Share of all savings %": round(tot.max() / tot.sum() * 100, 1)})
robust = pd.DataFrame(rows)
display(robust)

**How to read this:** under every like-for-like scheme the winning band is the one covering roughly the **late 20s to mid 30s**, so the conclusion does not hinge on where the band edges are drawn. Note that average balances are fairly **flat across age bands** (roughly LKR 0.7-0.8M), so the total is mostly a *head-count* effect (many customers in their late 20s and 30s) rather than older customers being richer.

---
# Q1C: Growing vs draining savings

**Definition:** a customer is **Growing** if monthly net flow (deposits − withdrawals) > 0, **Draining** if < 0, **Stable** if = 0. This is a *proxy* for balance growth (we only have a single-month average, not month-over-month balances, and interest is ignored).

The source `Net_Monthly_Flow` is used because it was verified against deposits − withdrawals on every valid row (see Section 0.6).

In [ ]:
flow = df["Net_Monthly_Flow"]
df["Flow_Class"] = np.select([flow > 0, flow < 0, flow == 0], ["Growing", "Draining", "Stable"], default="Unknown")
fc = df["Flow_Class"].value_counts().reindex(["Growing", "Draining", "Stable", "Unknown"], fill_value=0)
valid_n = fc[["Growing", "Draining", "Stable"]].sum()
flow_report = pd.DataFrame({"customers": fc, "pct_of_classifiable": (fc / valid_n * 100).round(2)})
flow_report.loc["Unknown", "pct_of_classifiable"] = np.nan
display(flow_report)
print(f"Classifiable customers: {valid_n} of {len(df)}")

**Important:** if 100% of customers are "growing", that is the **honest result** but it is *not* informative on its own. Customers with positive net flow cannot differ from each other on a growing/draining split, so it cannot help leadership prioritise. A bank's real concern is **how much cushion** each customer has. So I have gone one level deeper with two extra measures:
1. **Withdrawal-to-deposit ratio**: share of incoming money that goes straight back out. Above ~80% means a thin cushion (*fragile growers*).
2. **Monthly savings rate**: net flow ÷ balance, i.e. how fast the balance grows per month.

In [ ]:
df["Withdrawal_Ratio"] = df["Monthly_Withdrawal_Avg"] / df["Monthly_Deposit_Avg"]
df["Cushion_Band"] = pd.cut(df["Withdrawal_Ratio"], [0, 0.5, 0.8, np.inf],
                            labels=["Strong (<50% withdrawn)", "Moderate (50-80%)", "Fragile (>80% withdrawn)"])
cush = df["Cushion_Band"].value_counts().reindex(df["Cushion_Band"].cat.categories)
cush_tbl = pd.DataFrame({"customers": cush, "pct": (cush / cush.sum() * 100).round(1)})
display(cush_tbl)

df["Monthly_Savings_Rate_%"] = df["Net_Monthly_Flow"] / df["Savings_Balance"] * 100
print("Monthly savings rate (% of balance), distribution:")
display(df["Monthly_Savings_Rate_%"].describe().round(2).to_frame("value").T)

print("\nShare of FRAGILE growers by segment:")
seg_cushion = pd.crosstab(df["Customer_Segment"], df["Cushion_Band"], normalize="index").mul(100).round(1)
display(seg_cushion)
print("Share of FRAGILE growers by account status (note: Inactive/Suspended accounts still report deposits and withdrawals):")
display(pd.crosstab(df["Account_Status"], df["Cushion_Band"], normalize="index").mul(100).round(1))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(cush_tbl.index.astype(str), cush_tbl["customers"], color=["#2e8b57", "#e0a800", "#c0392b"])
ax[0].set_title("Savings cushion: how much of deposits is withdrawn"); ax[0].tick_params(axis="x", rotation=15)
ax[1].hist(df["Withdrawal_Ratio"].dropna(), bins=25, color="#1f4e79"); ax[1].axvline(0.8, color="#c0392b", ls="--")
ax[1].set_title("Withdrawal ÷ deposit ratio (red = 80%)")
plt.tight_layout(); plt.savefig(OUT / "q1c_flow_cushion.png", dpi=150); plt.show()

In [ ]:
grow_pct = fc["Growing"] / valid_n * 100
frag = cush_tbl["pct"].iloc[-1]
print(f"{grow_pct:.1f}% of classifiable customers have positive net monthly flow; {fc['Draining']} are draining.")
print(f"But {frag:.0f}% withdraw more than 80% of what they deposit, so their savings growth is thin.")
print(f"Median monthly savings rate is {df['Monthly_Savings_Rate_%'].median():.1f}% of balance (~{df['Monthly_Savings_Rate_%'].median()*12:.0f}% per year before interest).")

 *No customer is currently shrinking their savings, which is a positive but implausibly clean result. It is more likely a feature of how the data was compiled than proof that nobody ever draws savings down. The meaningful warning sign is a minority of customers who keep only a small slice of what they deposit, because a single bad month (job loss, illness) would turn them into drainers. That group is the one to monitor and support.*

**Caution to state:** Inactive and Suspended accounts still show monthly deposits and transactions, which is contradictory. So the flow figures describe *reported averages*, not verified activity.

---
## Q1 export: tables and charts

In [ ]:
with pd.ExcelWriter(OUT / "Q1_Workings.xlsx", engine="openpyxl") as xw:
    pd.DataFrame(cleaning_log).to_excel(xw, sheet_name="Cleaning_Log", index=False)
    for col, name in [("Customer_Segment", "Q1A_Segment"), ("Province", "Q1A_Province"), ("Urban_Rural", "Q1A_UrbanRural")]:
        q1a[col].round(2).to_excel(xw, sheet_name=name)
    age.round(2).to_excel(xw, sheet_name="Q1B_AgeBands")
    robust.to_excel(xw, sheet_name="Q1B_Robustness", index=False)
    flow_report.to_excel(xw, sheet_name="Q1C_Flow")
    cush_tbl.to_excel(xw, sheet_name="Q1C_Cushion")
    seg_cushion.to_excel(xw, sheet_name="Q1C_Cushion_by_Segment")
    for ws in xw.book.worksheets:
        ws.freeze_panes = "A2"
        for colx in ws.columns:
            w = max(len(str(c.value)) if c.value is not None else 0 for c in colx[:40])
            ws.column_dimensions[colx[0].column_letter].width = min(max(w + 2, 10), 55)

df.drop(columns=["Full_Name", "NIC_Number"], errors="ignore").to_csv(OUT / "FinSight_Q1_clean.csv", index=False)
print("Saved to", OUT.resolve(), ":", sorted(p.name for p in OUT.iterdir()))

---
# Q2. Loan portfolio and risk
Q2A loan penetration (overall, by segment and channel) · Q2B debt-to-savings ratio · Q2C default patterns · Q2D sensitivity analysis.

### Definitions used
| Metric | Definition | Denominator |
|---|---|---|
| Loan penetration | Customers holding a loan ÷ all customers | 500 de-duplicated customers |
| Debt-to-Savings Ratio (DSR) | Outstanding loan balance ÷ savings balance | Loan customers with complete data **and** a valid, positive savings balance |
| Exposure | Sum of **outstanding** balances (not original loan amounts) | n/a |
| Default rate | Loans with status "Defaulted" ÷ loans with complete data | 218 loans |
| Problem-loan (delinquency) rate | "Delayed" **or** "Defaulted" ÷ loans with complete data | 218 loans |

**Statistical warning for this question:** there are only about **22 defaults**. Any single segment, district or loan-type cell contains very few, so apparent patterns can easily be chance. The analysis therefore shows **confidence intervals and permutation tests**, and labels groups *Elevated*, *Watch* or *Normal* with an explicit rule instead of overstating what 22 events can prove.

In [ ]:
OUT = OUT_ROOT / "q2"; OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

---
# Q2A: Loan penetration
**Penetration = customers holding a loan ÷ all customers.** "Concentration" can mean two different things, so both are shown:
1. **Penetration rate**: within the group, what share has a loan? (Where is lending *most intense*?)
2. **Share of the loan book**: what share of all loan customers / all outstanding LKR sits in the group? (Where is the *money*?)

Each rate carries a **Wilson 95% confidence interval** and the difference between groups is tested with a chi-square test.

In [ ]:
n_all, n_loans = len(df), int(df["Loan_Holder"].sum())
pen_all = n_loans / n_all
lo, hi = stats.binomtest(n_loans, n_all).proportion_ci(method="wilson")
print(f"OVERALL loan penetration: {n_loans} of {n_all} customers = {pen_all:.1%}  (95% CI {lo:.1%} to {hi:.1%})")

def pen_table(col):
    g = df.groupby(col)
    t = g.agg(customers=("Customer_ID", "size"), loan_customers=("Loan_Holder", "sum"), outstanding=("Outstanding_Loan_Balance", "sum"))
    t["penetration_%"] = t["loan_customers"] / t["customers"] * 100
    ci = [stats.binomtest(int(k), int(n)).proportion_ci(method="wilson") for k, n in zip(t["loan_customers"], t["customers"])]
    t["ci_low_%"] = [c.low * 100 for c in ci]; t["ci_high_%"] = [c.high * 100 for c in ci]
    t["share_of_loan_customers_%"] = t["loan_customers"] / t["loan_customers"].sum() * 100
    t["share_of_loan_book_%"] = t["outstanding"] / t["outstanding"].sum() * 100
    t["avg_outstanding_per_loan"] = t["outstanding"] / t["loan_customers"]
    return t.sort_values("penetration_%", ascending=False)

q2a = {}; q2a_p = {}
for col in ["Customer_Segment", "Acquisition_Channel"]:
    q2a[col] = pen_table(col)
    q2a_p[col] = stats.chi2_contingency(pd.crosstab(df[col], df["Loan_Holder"]))[1]
    print(f"\n=== {col} | chi-square p = {q2a_p[col]:.3f} ===")
    display(q2a[col].round({"outstanding": 0, "penetration_%": 1, "ci_low_%": 1, "ci_high_%": 1, "share_of_loan_customers_%": 1,
                            "share_of_loan_book_%": 1, "avg_outstanding_per_loan": 0}))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, col in zip(ax[:2], ["Customer_Segment", "Acquisition_Channel"]):
    t = q2a[col]
    err = [t["penetration_%"] - t["ci_low_%"], t["ci_high_%"] - t["penetration_%"]]
    a.bar(t.index, t["penetration_%"], yerr=err, capsize=4, color="#1f4e79")
    a.axhline(pen_all * 100, color="#c0392b", ls="--"); a.set_ylim(0, 70)
    a.set_title(f"Loan penetration by {col.replace('_', ' ')} (bars = 95% CI)")
    for i, v in enumerate(t["penetration_%"]): a.text(i, 2, f"{v:.0f}%", ha="center", color="white", fontsize=9)
t = q2a["Customer_Segment"]
ax[2].bar(t.index, t["share_of_loan_book_%"], color="#2e8b57"); ax[2].set_title("Share of outstanding loan book by segment (%)")
for i, v in enumerate(t["share_of_loan_book_%"]): ax[2].text(i, v + 1, f"{v:.0f}%", ha="center")
plt.tight_layout(); plt.savefig(OUT / "q2a_penetration.png", dpi=150); plt.show()

**Sensitivity of the headline to the loan-holder rule:** the penetration figure barely moves under reasonable alternative definitions.

In [ ]:
alt = pd.DataFrame({
    "Definition": ["Base rule (flag OR details)", "Strict: Has_Loan = 'Yes' only", "Only loans with complete data"],
    "Loan customers": [int(df["Loan_Holder"].sum()), int((df["Has_Loan"] == "Yes").sum()), int(df["Loan_Data_Complete"].sum())],
})
alt["Penetration_%"] = (alt["Loan customers"] / n_all * 100).round(1)
display(alt)
print("Customers whose loan status is unknown (Has_Loan blank, no details):", int(df["Loan_Status_Unknown"].sum()),
      "-> penetration if all of them did hold a loan:", f"{(n_loans + df['Loan_Status_Unknown'].sum())/n_all:.1%}")

In [ ]:
seg, ch = q2a["Customer_Segment"], q2a["Acquisition_Channel"]
print(f"HEADLINE: {pen_all:.1%} of customers hold a loan.")
print(f"SEGMENT : penetration is {seg['penetration_%'].min():.0f}% to {seg['penetration_%'].max():.0f}% (chi-square p = {q2a_p['Customer_Segment']:.2f}) -> "
      + ("a real difference." if q2a_p['Customer_Segment'] < .05 else "NO statistically meaningful difference in how likely each segment is to borrow."))
print(f"          But the MONEY is concentrated: {seg['share_of_loan_book_%'].idxmax()} holds {seg['share_of_loan_book_%'].max():.0f}% of the outstanding book, simply because it is the largest segment.")
print(f"CHANNEL : penetration is {ch['penetration_%'].min():.0f}% ({ch['penetration_%'].idxmin()}) to {ch['penetration_%'].max():.0f}% ({ch['penetration_%'].idxmax()}) (p = {q2a_p['Acquisition_Channel']:.2f}) -> "
      + ("a real difference." if q2a_p['Acquisition_Channel'] < .05 else "directional only: Agent-acquired customers borrow the most, but the gap is within sampling noise."))
print(f"          Largest share of the loan book by channel: {ch['share_of_loan_book_%'].idxmax()} ({ch['share_of_loan_book_%'].max():.0f}%).")

---
# Q2B: Debt-to-Savings Ratio (DSR)
**DSR = Outstanding Loan Balance ÷ Savings Balance**, for customers with loans. A DSR above 1.0 means the customer owes more than they hold in savings, so a default would not be covered by their deposits.

**Population:** loans with complete data **and** a valid, positive savings balance. Customers whose savings were invalid/missing are listed separately (their exposure is *not assessable*), so nothing is silently dropped.

In [ ]:
LD = df[df["Loan_Data_Complete"]].copy()
LD["DSR"] = LD["Outstanding_Loan_Balance"] / LD["Savings_Balance"].where(LD["Savings_Balance"] > 0)
assess = LD[LD["DSR"].notna()].copy()
not_assess = LD[LD["DSR"].isna()]

hi = assess[assess["DSR"] > 1]
n_hi, exp_hi = len(hi), hi["Outstanding_Loan_Balance"].sum()
book_assess = assess["Outstanding_Loan_Balance"].sum()

print(f"Loans with complete data: {len(LD)} | DSR assessable: {len(assess)} | not assessable (savings invalid/missing): {len(not_assess)}")
print(f"\n>>> ANSWER: {n_hi} customers have DSR > 1.0  ({n_hi/len(assess):.1%} of assessable loan customers)")
print(f">>> Total LKR exposure of those customers (outstanding balances): LKR {exp_hi:,.0f}  ({exp_hi/1e6:,.1f} million)")
print(f"    = {exp_hi/book_assess:.1%} of the assessable loan book")
print(f"\nMedian DSR: {assess['DSR'].median():.2f} | mean: {assess['DSR'].mean():.2f} (mean is distorted by a few extreme ratios, max {assess['DSR'].max():.0f})")
print(f"Not assessable: outstanding LKR {not_assess['Outstanding_Loan_Balance'].sum():,.0f} across {len(not_assess)} customers")
display(not_assess[["Customer_ID", "Customer_Segment", "Loan_Type", "Outstanding_Loan_Balance", "Savings_Balance"]])

In [ ]:
bands = pd.cut(assess["DSR"], [0, 0.5, 1, 2, 5, np.inf], labels=["≤0.5x", "0.5-1x", "1-2x", "2-5x", ">5x"])
dsr_band = (assess.groupby(bands, observed=False)
            .agg(customers=("DSR", "size"), outstanding=("Outstanding_Loan_Balance", "sum")))
dsr_band["% customers"] = dsr_band["customers"] / dsr_band["customers"].sum() * 100
dsr_band["% of loan book"] = dsr_band["outstanding"] / dsr_band["outstanding"].sum() * 100
display(dsr_band.round({"outstanding": 0, "% customers": 1, "% of loan book": 1}))

def dsr_by(col):
    g = assess.groupby(col)
    t = g.agg(loans=("DSR", "size"), median_DSR=("DSR", "median"),
              over_1=("DSR", lambda s: int((s > 1).sum())), outstanding=("Outstanding_Loan_Balance", "sum"))
    t["% over_1"] = t["over_1"] / t["loans"] * 100
    t["exposure_over_1"] = g.apply(lambda x: x.loc[x["DSR"] > 1, "Outstanding_Loan_Balance"].sum(), include_groups=False)
    return t

dsr_seg, dsr_type = dsr_by("Customer_Segment"), dsr_by("Loan_Type")
print("By segment:"); display(dsr_seg.round({"median_DSR": 2, "% over_1": 1, "outstanding": 0, "exposure_over_1": 0}))
print("By loan type:"); display(dsr_type.round({"median_DSR": 2, "% over_1": 1, "outstanding": 0, "exposure_over_1": 0}))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].bar(dsr_band.index.astype(str), dsr_band["customers"], color=["#2e8b57", "#7fb77e", "#e0a800", "#e67e22", "#c0392b"])
ax[0].set_title("Loan customers by DSR band")
ax[1].bar(dsr_band.index.astype(str), dsr_band["outstanding"], color=["#2e8b57", "#7fb77e", "#e0a800", "#e67e22", "#c0392b"])
ax[1].yaxis.set_major_formatter(FuncFormatter(lkr_fmt)); ax[1].set_title("Outstanding LKR by DSR band")
ax[2].bar(dsr_seg.index, dsr_seg["% over_1"], color="#1f4e79"); ax[2].set_title("% of loan customers with DSR > 1, by segment")
for i, v in enumerate(dsr_seg["% over_1"]): ax[2].text(i, v + 1, f"{v:.0f}%", ha="center")
plt.tight_layout(); plt.savefig(OUT / "q2b_dsr.png", dpi=150); plt.show()

**Reading the debt-to-savings ratio correctly:**
- DSR > 1 is **driven by small savings balances more than by large loans.** Starter-segment customers have tiny balances, so nearly all of them exceed 1.0 (see the segment table), whereas only a small minority of Premium customers do. DSR therefore partly re-measures *segment* rather than independent risk.
- DSR compares a loan to *savings only*. Fixed deposits appear to be held **outside** the savings balance (the FD amount exceeds the savings balance for most FD holders, counted in Q2D), so DSR **overstates** how exposed some customers are. Q2D re-runs it on savings + FD.
- A high DSR is a measure of **thin deposit cover**, not proof of default. The next cell checks whether DSR actually predicts defaults in this data.

In [ ]:
# Does a high DSR actually go with more defaults? (the thing a risk manager really cares about)
assess["is_def"] = assess["Loan_Repayment_Status"] == "Defaulted"
assess["is_prob"] = assess["Loan_Repayment_Status"].isin(["Defaulted", "Delayed"])
cmp = assess.groupby(assess["DSR"] > 1).agg(loans=("is_def", "size"), defaults=("is_def", "sum"), default_rate=("is_def", "mean"), problem_rate=("is_prob", "mean"))
cmp.index = ["DSR ≤ 1", "DSR > 1"]; cmp["default_rate"] *= 100; cmp["problem_rate"] *= 100
display(cmp.round(1))
tab = pd.crosstab(assess["DSR"] > 1, assess["is_def"])
odds, p_dsr_def = stats.fisher_exact(tab)
print(f"Fisher exact test (DSR>1 vs default): p = {p_dsr_def:.2f} ->",
      "DSR > 1 IS associated with defaults." if p_dsr_def < .05 else "no statistically reliable link between DSR>1 and default in this sample.")
by_band = assess.groupby(bands, observed=False).agg(loans=("is_def", "size"), defaults=("is_def", "sum"))
by_band["default_rate_%"] = by_band["defaults"] / by_band["loans"] * 100
display(by_band.round(1))

print("\nTop 10 individual exposures with DSR > 1 (de-identified: IDs only):")
top10 = hi.sort_values("Outstanding_Loan_Balance", ascending=False).head(10)[
    ["Customer_ID", "Customer_Segment", "Loan_Type", "Outstanding_Loan_Balance", "Savings_Balance", "DSR", "Loan_Repayment_Status"]]
display(top10.round({"Outstanding_Loan_Balance": 0, "Savings_Balance": 0, "DSR": 1}))

---
# Q2C: Default patterns (Segment, District, Loan Type)
**Default rate = Defaulted ÷ loans with complete data.** Because defaults are rare, every group gets:
- the raw count and rate,
- a **Wilson 95% CI** (how wide the plausible range is; wide = weak evidence),
- *expected* defaults if the group were average, and a one-sided **binomial p-value** for "worse than the portfolio",
- a **flag**: `ELEVATED` = rate ≥ 1.5× portfolio **and** ≥ 3 defaults; `WATCH` = rate ≥ 1.5× portfolio with exactly 2 defaults (too few events to act on); a group with a single default is never flagged; otherwise `-`.

An omnibus **permutation test** per dimension answers the key question: *is there any real pattern at all, or is this what random scatter looks like?*

In [ ]:
L = df[df["Loan_Data_Complete"]].copy()
L["is_def"] = L["Loan_Repayment_Status"] == "Defaulted"
L["is_prob"] = L["Loan_Repayment_Status"].isin(["Defaulted", "Delayed"])
L["def_exposure"] = np.where(L["is_def"], L["Outstanding_Loan_Balance"], 0.0)

n_loans_c, n_def = len(L), int(L["is_def"].sum())
P = n_def / n_loans_c
lo, hi_ = stats.binomtest(n_def, n_loans_c).proportion_ci(method="wilson")
print(f"PORTFOLIO DEFAULT RATE: {n_def} of {n_loans_c} loans = {P:.1%}  (95% CI {lo:.1%} to {hi_:.1%})")
print(f"PROBLEM-LOAN RATE (delayed or defaulted): {L['is_prob'].mean():.1%}  -> only {(~L['is_prob']).mean():.0%} of loans are fully on time")
print(f"Defaulted exposure: LKR {L['def_exposure'].sum():,.0f} = {L['def_exposure'].sum()/L['Outstanding_Loan_Balance'].sum():.1%} of outstanding loan book")
print("Repayment status mix:"); display(L["Loan_Repayment_Status"].value_counts().to_frame("loans").assign(pct=lambda x: (x["loans"]/x["loans"].sum()*100).round(1)))

In [ ]:
def rate_table(cols):
    g = L.groupby(cols)
    t = g.agg(loans=("is_def", "size"), defaults=("is_def", "sum"), problem_loans=("is_prob", "sum"),
              outstanding=("Outstanding_Loan_Balance", "sum"), default_exposure=("def_exposure", "sum"))
    t["default_rate_%"] = t["defaults"] / t["loans"] * 100
    ci = [stats.binomtest(int(d), int(n)).proportion_ci(method="wilson") for d, n in zip(t["defaults"], t["loans"])]
    t["ci_low_%"] = [c.low * 100 for c in ci]; t["ci_high_%"] = [c.high * 100 for c in ci]
    t["expected_defaults"] = t["loans"] * P
    t["p_worse_than_portfolio"] = [stats.binomtest(int(d), int(n), P, alternative="greater").pvalue for d, n in zip(t["defaults"], t["loans"])]
    t["problem_rate_%"] = t["problem_loans"] / t["loans"] * 100
    hot = t["default_rate_%"] >= ELEVATED_MULT * P * 100
    t["flag"] = np.where(hot & (t["defaults"] >= ELEVATED_MIN_DEFAULTS), "ELEVATED",
                         np.where(hot & (t["defaults"] == 2), "WATCH (only 2 defaults)", "-"))   # 1 default is anecdote: never flagged
    return t.sort_values("default_rate_%", ascending=False)

def perm_p(cols, n_perm=N_PERM):
    """Omnibus permutation test: is variation in default rate across groups bigger than random scatter?
    Loans with a missing grouping value (e.g. blank District) are excluded, exactly as in rate_table()."""
    sub = L.dropna(subset=cols if isinstance(cols, list) else [cols])
    p0 = sub["is_def"].mean()
    g = sub.groupby(cols).ngroup().values.astype(int); k = g.max() + 1
    y = sub["is_def"].astype(int).values; n = np.bincount(g, minlength=k)
    def stat(yy):
        d = np.bincount(g, weights=yy, minlength=k)
        return (((d - n * p0) ** 2) / (n * p0 * (1 - p0))).sum()
    obs = stat(y)
    sims = np.array([stat(rng.permutation(y)) for _ in range(n_perm)])
    return (sims >= obs).mean()

miss_dist = int(L["District"].isna().sum())
print(f"Note: {miss_dist} loan(s) have a blank District and are excluded from District tables only (kept in all other analyses).")

q2c, q2c_p = {}, {}
for col in ["Customer_Segment", "Loan_Type", "District"]:
    q2c[col] = rate_table(col); q2c_p[col] = perm_p(col)
    print(f"\n=== {col} | omnibus permutation p = {q2c_p[col]:.3f} ===")
    display(q2c[col][["loans", "defaults", "default_rate_%", "ci_low_%", "ci_high_%", "expected_defaults", "p_worse_than_portfolio", "problem_rate_%", "default_exposure", "flag"]]
            .round({"default_rate_%": 1, "ci_low_%": 1, "ci_high_%": 1, "expected_defaults": 1, "p_worse_than_portfolio": 3, "problem_rate_%": 1, "default_exposure": 0}))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, col in zip(ax, ["Customer_Segment", "Loan_Type", "District"]):
    t = q2c[col].sort_values("default_rate_%")
    y = np.arange(len(t))
    colors = ["#c0392b" if f == "ELEVATED" else "#e0a800" if str(f).startswith("WATCH") else "#1f4e79" for f in t["flag"]]
    a.barh(y, t["default_rate_%"], color=colors)
    a.errorbar(t["default_rate_%"], y, xerr=[t["default_rate_%"] - t["ci_low_%"], t["ci_high_%"] - t["default_rate_%"]], fmt="none", ecolor="black", capsize=3, lw=1)
    a.axvline(P * 100, color="grey", ls="--"); a.set_yticks(y); a.set_yticklabels([f"{i} (n={n})" for i, n in zip(t.index, t["loans"])])
    a.set_title(f"Default rate by {col.replace('_', ' ')}\n(red=elevated, amber=watch; whiskers = 95% CI)", fontsize=9); a.set_xlabel("%")
plt.tight_layout(); plt.savefig(OUT / "q2c_default_rates.png", dpi=150); plt.show()

**Combinations.** The question asks for *combinations* that are elevated. With ~218 loans, a Segment × Loan Type grid has 12 cells (about 18 loans each) and a District × Loan Type grid has 40 cells (about 5 loans each). The flags below follow exactly the same rule, and every cell shows its size so nobody over-reads a "30% default rate" that is 4 defaults out of 13.

In [ ]:
seg_type = rate_table(["Customer_Segment", "Loan_Type"])
print("Segment x Loan Type  (omnibus permutation p = %.3f)" % perm_p(["Customer_Segment", "Loan_Type"]))
display(seg_type[["loans", "defaults", "default_rate_%", "ci_low_%", "ci_high_%", "problem_rate_%", "p_worse_than_portfolio", "flag"]]
        .round({"default_rate_%": 1, "ci_low_%": 1, "ci_high_%": 1, "problem_rate_%": 1, "p_worse_than_portfolio": 3}))

# heat-map: default rate with "defaults/loans" in each cell
piv_r = L.pivot_table(index="Customer_Segment", columns="Loan_Type", values="is_def", aggfunc="mean") * 100
piv_d = L.pivot_table(index="Customer_Segment", columns="Loan_Type", values="is_def", aggfunc="sum")
piv_n = L.pivot_table(index="Customer_Segment", columns="Loan_Type", values="is_def", aggfunc="count")
fig, ax = plt.subplots(figsize=(7, 3.4))
im = ax.imshow(piv_r.values, cmap="Reds", vmin=0, vmax=35)
ax.set_xticks(range(len(piv_r.columns))); ax.set_xticklabels(piv_r.columns); ax.set_yticks(range(len(piv_r.index))); ax.set_yticklabels(piv_r.index)
for i in range(piv_r.shape[0]):
    for j in range(piv_r.shape[1]):
        ax.text(j, i, f"{piv_r.iloc[i, j]:.0f}%\n({int(piv_d.iloc[i, j])}/{int(piv_n.iloc[i, j])})", ha="center", va="center", fontsize=9)
ax.set_title("Default rate: Segment x Loan Type  (cell shows defaults/loans)"); plt.colorbar(im, ax=ax, label="%")
plt.tight_layout(); plt.savefig(OUT / "q2c_heatmap.png", dpi=150); plt.show()

In [ ]:
dist_type = rate_table(["District", "Loan_Type"])
print("District x Loan Type: %d cells, median loans per cell = %.0f, omnibus permutation p = %.3f" % (len(dist_type), dist_type["loans"].median(), perm_p(["District", "Loan_Type"])))
print("Cells with at least 2 defaults (everything else has 0-1 defaults: noise):")
display(dist_type[dist_type["defaults"] >= 2][["loans", "defaults", "default_rate_%", "ci_low_%", "ci_high_%", "flag"]].round(1))

In [ ]:
# ---- consolidated watch-list ---------------------------------------------------------------------------
frames = []
for col in ["Customer_Segment", "Loan_Type", "District"]:
    t = q2c[col].reset_index().rename(columns={col: "group"}); t.insert(0, "dimension", col); frames.append(t)
for name, t in [("Segment x Loan Type", seg_type), ("District x Loan Type", dist_type)]:
    t = t.reset_index(); t["group"] = t.iloc[:, 0].astype(str) + " | " + t.iloc[:, 1].astype(str); t.insert(0, "dimension", name); frames.append(t)
allt = pd.concat(frames, ignore_index=True)
watch = allt[allt["flag"] != "-"][["dimension", "group", "loans", "defaults", "default_rate_%", "ci_low_%", "ci_high_%", "p_worse_than_portfolio", "problem_rate_%", "flag"]]
watch = watch.sort_values(["flag", "default_rate_%"], ascending=[True, False])
print("WATCH-LIST (default rate >= %.1fx portfolio rate of %.1f%%):" % (ELEVATED_MULT, P * 100))
display(watch.round({"default_rate_%": 1, "ci_low_%": 1, "ci_high_%": 1, "p_worse_than_portfolio": 3, "problem_rate_%": 1}))
print("\nBest single reading of the evidence:")
for col in ["Customer_Segment", "Loan_Type", "District"]:
    print(f"  {col:17s}: omnibus p = {q2c_p[col]:.2f} ->", "real pattern" if q2c_p[col] < .05 else "indistinguishable from random scatter")
print("  Multiple-testing note: ~%d groups/cells were examined, so a few 'significant-looking' p-values are expected by chance alone." % len(allt))

**Reading Q2C honestly:**
1. The **flagged groups** answer the question "which combinations represent elevated risk".
2. The **omnibus tests find no reliable pattern**, so these are a **watch-list for monitoring**, not proof of a risk driver.
3. The **bigger, better-supported finding is:** the **problem-loan rate** (delayed + defaulted) is far higher than the default rate, and delinquencies are numerous enough to analyse. Only about 6 in 10 loans are fully on time.

---
# Q2D: Sensitivity analysis: do the headline answers survive alternative choices?
A conclusion that flips when you tweak one assumption is fragile. This table re-computes the three headline numbers under alternative rules.

In [ ]:
def dsr_stats(sub, use_fd=False):
    a = sub["Savings_Balance"].where(sub["Savings_Balance"] > 0)
    if use_fd: a = a + sub["FD_Amount"].fillna(0)
    r = sub["Outstanding_Loan_Balance"] / a
    ok = r.notna(); hi_ = ok & (r > 1)
    return int(ok.sum()), int(hi_.sum()), float(sub.loc[hi_, "Outstanding_Loan_Balance"].sum())

def def_stats(sub):
    d = (sub["Loan_Repayment_Status"] == "Defaulted"); return int(d.sum()), len(sub), d.mean() * 100

base = df[df["Loan_Data_Complete"]]
noconf = base[~base["Flag_Loan_Date_Conflict"]]
strict = base[base["Has_Loan"] == "Yes"]
v71 = df[df["Loan_Holder"] & df["Loan_Repayment_Status"].notna()]            # adds FSL-0071 (Defaulted, no loan type)

rows = []
def add(name, pen_n, sub, fd=False, dsub=None):
    n_ok, n_hi_, exp_ = dsr_stats(sub, fd)
    dd, dn, dr = def_stats(dsub if dsub is not None else sub)
    rows.append({"Scenario": name, "Penetration %": round(pen_n / n_all * 100, 1), "DSR>1 customers": n_hi_, "DSR>1 of assessable": n_ok,
                 "DSR>1 exposure (LKR m)": round(exp_ / 1e6, 1), "Default rate %": round(dr, 1), "Defaults / loans": f"{dd}/{dn}"})
add("BASE (rules in Section 0.10)", df["Loan_Holder"].sum(), base)
add("Strict: Has_Loan='Yes' only", (df["Has_Loan"] == "Yes").sum(), strict)
add("Exclude loans with date conflicts", df["Loan_Holder"].sum() - base["Flag_Loan_Date_Conflict"].sum(), noconf)
add("DSR on savings + Fixed Deposit", df["Loan_Holder"].sum(), base, fd=True)
add("Also count FSL-0071 (defaulted, no loan type)", df["Loan_Holder"].sum(), base, dsub=v71)
sens = pd.DataFrame(rows); display(sens)
fd_holders = df[df["FD_Amount"].notna() & df["Savings_Balance"].notna()]
print(f"FD_Amount exceeds Savings_Balance for {(fd_holders['FD_Amount'] > fd_holders['Savings_Balance']).sum()} of {len(fd_holders)} FD holders "
      "-> FDs sit outside the savings balance, so a savings-only DSR overstates exposure.")

**Interpretation:** penetration and default rate barely move under any alternative. The DSR>1 count *does* move when fixed deposits are counted as customer assets, so the honest statement is "between roughly 100 and 127 customers, depending on whether FDs are counted". Present the base answer and quote the range.

---
## Key findings (generated from the numbers) and export

In [ ]:
seg_pen, ch_pen = q2a["Customer_Segment"], q2a["Acquisition_Channel"]
elev = watch[watch["flag"] == "ELEVATED"]
print("1) PENETRATION  :", f"{pen_all:.1%} of customers hold a loan; no meaningful difference by segment (p={q2a_p['Customer_Segment']:.2f}) or channel (p={q2a_p['Acquisition_Channel']:.2f}).")
print("                  Most loan LKR sits in", seg_pen['share_of_loan_book_%'].idxmax(), f"({seg_pen['share_of_loan_book_%'].max():.0f}% of the book) because it is the largest segment.")
print("2) DSR > 1      :", f"{n_hi} customers ({n_hi/len(assess):.0%} of assessable) owe more than they save, LKR {exp_hi/1e6:,.1f}m ({exp_hi/book_assess:.0%} of the assessable book).")
print(f"                  Range {sens['DSR>1 customers'].min()}-{sens['DSR>1 customers'].max()} customers depending on whether FDs count. Almost entirely a Starter/Regular small-balance effect; not a significant predictor of default (p={p_dsr_def:.2f}).")
print("3) DEFAULTS     :", f"{P:.1%} default rate ({n_def}/{n_loans_c}), but {L['is_prob'].mean():.0%} of loans are delayed or defaulted.")
print("                  Watch-list:", "; ".join(f"{r.group} ({int(r.defaults)}/{int(r.loans)} = {r['default_rate_%']:.0f}%)" for _, r in elev.iterrows()) if len(elev) else "none")
print("                  No dimension shows a statistically reliable pattern, so these are monitoring flags, not conclusions.")

In [ ]:
tables = {
    "Cleaning_Log": pd.DataFrame(cleaning_log),
    "Q2A_Segment": q2a["Customer_Segment"], "Q2A_Channel": q2a["Acquisition_Channel"], "Q2A_Definitions": alt,
    "Q2B_DSR_Bands": dsr_band, "Q2B_DSR_Segment": dsr_seg, "Q2B_DSR_LoanType": dsr_type, "Q2B_DSR_vs_Default": by_band, "Q2B_Top10": top10,
    "Q2C_Segment": q2c["Customer_Segment"], "Q2C_LoanType": q2c["Loan_Type"], "Q2C_District": q2c["District"],
    "Q2C_Seg_x_Type": seg_type, "Q2C_Dist_x_Type": dist_type, "Q2C_Watchlist": watch, "Sensitivity": sens,
}
with pd.ExcelWriter(OUT / "Q2_Workings.xlsx", engine="openpyxl") as xw:
    for name, t in tables.items():
        keep_index = isinstance(t.index, pd.MultiIndex) or t.index.name is not None or name in ("Q2B_DSR_Bands", "Q2B_DSR_vs_Default")
        t.round(3).to_excel(xw, sheet_name=name[:31], index=keep_index)
    for ws in xw.book.worksheets:
        ws.freeze_panes = "A2"
        for colx in ws.columns:
            w = max(len(str(c.value)) if c.value is not None else 0 for c in colx[:40])
            ws.column_dimensions[colx[0].column_letter].width = min(max(w + 2, 10), 50)
print("Saved to", OUT.resolve(), ":", sorted(p.name for p in OUT.iterdir()))

In [ ]:
q2_sens = sens.copy()          # Q2D sensitivity table, kept under its own name (Q4.4 reuses the name `sens`)

---
# Q3. Digital engagement and product holdings
Q3A do mobile-app users hold higher savings? · Q3B cross-sell opportunity (savings but no Fixed Deposit and no Insurance) · Q3C which acquisition channel produces the best customers?

**Approach:** separate *statistical* significance, *practical* size and *causal* meaning; check for confounders (segment, age, location); test alternative definitions where a flag is unreliable; and say "no meaningful difference" where that is what the data shows.

In [ ]:
OUT = OUT_ROOT / "q3"; OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

---
# Q3A: Do mobile-app users hold higher savings?

**Approach**
1. Compare the two groups on mean *and* median (balances are skewed, so the median is the fairer centre).
2. Test the difference two ways (Welch t-test on means; Mann-Whitney on ranks) and bootstrap the size of the gap with a 95% confidence interval.
3. **Check for confounding:** compare *within* segment, then regress log-savings on app use with and without controls.
4. **Check the definition:** the app flag contradicts other columns for some customers, so test alternative definitions of "app user".
5. Conclude on three separate questions: is it **statistically** significant, **practically** meaningful, and **causal**?

In [ ]:
S = df.dropna(subset=["Savings_Balance"]).copy()
S["App"] = np.where(S["Mobile_App_User"] == "Yes", "App user", "Non-user")
print(f"App adoption: {(df['Mobile_App_User']=='Yes').mean():.1%} of all {len(df)} customers  |  customers with a valid balance: {len(S)}")

grp = S.groupby("App")["Savings_Balance"].agg(customers="count", mean="mean", median="median", std="std")
grp["total"] = S.groupby("App")["Savings_Balance"].sum()
display(grp.round(0))
a = S.loc[S["App"] == "App user", "Savings_Balance"].values
b = S.loc[S["App"] == "Non-user", "Savings_Balance"].values
print(f"\nMean:   app {a.mean():,.0f} vs non-user {b.mean():,.0f}  -> app users are {a.mean()/b.mean()-1:+.1%}")
print(f"Median: app {np.median(a):,.0f} vs non-user {np.median(b):,.0f}  -> app users are {np.median(a)/np.median(b)-1:+.1%}")

In [ ]:
# ---- significance, effect size and bootstrap CI of the gap ---------------------------------------
t_res = stats.ttest_ind(a, b, equal_var=False)
u_res = stats.mannwhitneyu(a, b, alternative="two-sided")
cliff = 2 * u_res.statistic / (len(a) * len(b)) - 1          # Cliff's delta: P(app > non) - P(app < non); 0 = no effect
print(f"Welch t-test on means:    p = {t_res.pvalue:.3f}")
print(f"Mann-Whitney on ranks:    p = {u_res.pvalue:.3f}")
print(f"Cliff's delta (effect size): {cliff:+.3f}   (|d| < 0.15 is negligible)")

boot_mean, boot_med = np.empty(N_BOOT), np.empty(N_BOOT)
for i in range(N_BOOT):
    ra, rb = rng.choice(a, len(a)), rng.choice(b, len(b))
    boot_mean[i], boot_med[i] = ra.mean() - rb.mean(), np.median(ra) - np.median(rb)
ci_mean, ci_med = np.percentile(boot_mean, [2.5, 97.5]), np.percentile(boot_med, [2.5, 97.5])
print(f"\nBootstrap 95% CI for (app - non-user):")
print(f"   mean   difference: LKR {a.mean()-b.mean():+,.0f}   [{ci_mean[0]:+,.0f}, {ci_mean[1]:+,.0f}]")
print(f"   median difference: LKR {np.median(a)-np.median(b):+,.0f}   [{ci_med[0]:+,.0f}, {ci_med[1]:+,.0f}]")
print("   -> the interval spans zero: the data cannot tell us app users hold more." if ci_mean[0] < 0 < ci_mean[1] else "   -> interval excludes zero.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].boxplot([a, b], tick_labels=["App user", "Non-user"], showfliers=True)
ax[0].set_yscale("log"); ax[0].set_title("Savings distribution (log scale)"); ax[0].yaxis.set_major_formatter(FuncFormatter(lkr_fmt))
x = np.arange(2); w = 0.35
ax[1].bar(x - w/2, [a.mean(), b.mean()], w, label="Mean", color="#1f4e79"); ax[1].bar(x + w/2, [np.median(a), np.median(b)], w, label="Median", color="#2e8b57")
ax[1].set_xticks(x); ax[1].set_xticklabels(["App user", "Non-user"]); ax[1].yaxis.set_major_formatter(FuncFormatter(lkr_fmt)); ax[1].legend(); ax[1].set_title("Mean vs median savings")
ax[2].hist([np.log10(a), np.log10(b)], bins=20, label=["App user", "Non-user"], color=["#1f4e79", "#e0a800"], density=True)
ax[2].set_title("Distribution of log10(savings)"); ax[2].legend()
plt.tight_layout(); plt.savefig(OUT / "q3a_app_vs_savings.png", dpi=150); plt.show()

### Confounding check 1: compare *within* each segment
If app users are concentrated in a particular segment, a pooled comparison is distorted. Comparing inside each segment removes that effect.

In [ ]:
print("App adoption by segment (%):")
display((pd.crosstab(df["Customer_Segment"], df["Mobile_App_User"], normalize="index") * 100).round(1))

rows = []
for sname, g in S.groupby("Customer_Segment"):
    ga, gb = g.loc[g["App"] == "App user", "Savings_Balance"], g.loc[g["App"] == "Non-user", "Savings_Balance"]
    rows.append({"Segment": sname, "n_app": len(ga), "n_non": len(gb), "median_app": ga.median(), "median_non": gb.median(),
                 "median_gap_%": (ga.median() / gb.median() - 1) * 100, "mean_gap_%": (ga.mean() / gb.mean() - 1) * 100,
                 "MWU_p": stats.mannwhitneyu(ga, gb).pvalue})
seg_app = pd.DataFrame(rows).set_index("Segment")
display(seg_app.round({"median_app": 0, "median_non": 0, "median_gap_%": 1, "mean_gap_%": 1, "MWU_p": 3}))
n_pos_med = int((seg_app["median_gap_%"] > 0).sum()); n_pos_mean = int((seg_app["mean_gap_%"] > 0).sum())
print(f"Reading: app users' MEDIAN is higher in {n_pos_med} of 3 segments ({seg_app['median_gap_%'].min():+.0f}% to {seg_app['median_gap_%'].max():+.0f}%), "
      f"but the MEAN is higher in only {n_pos_mean} of 3, and no segment gap is statistically reliable (smallest p = {seg_app['MWU_p'].min():.2f}).")
print("Premium shows the largest gap but has only %d app users and %d non-users: treat it as a hypothesis to test on more data, not a finding." % (seg_app.loc['Premium','n_app'], seg_app.loc['Premium','n_non']))
print("Why the pooled gap is ~0 while within-segment medians lean positive: the segments have different app adoption (Premium is the LEAST digital), which dilutes the pooled comparison.")

### Confounding check 2: regression on log-savings
`log(savings) = a + b·App + controls`. Then `exp(b) − 1` is the **% difference in savings associated with app use**, holding the controls fixed. Robust (HC1) standard errors are used.

Three models, deliberately:
1. **Unadjusted:** the raw comparison.
2. **Adjusted for independent traits:** age, tenure, gender, urban/rural (things that exist *before* and independently of balance).
3. **Plus segment:** shown with a **warning**. Segment is assigned *from* balance, so controlling for it partly controls for the outcome itself (a "bad control"). It is included only to show the result does not change.

In [ ]:
def ols_hc1(y, X):
    X = np.asarray(X, float); y = np.asarray(y, float)
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X); beta = XtX_inv @ X.T @ y; e = y - X @ beta
    meat = (X * e[:, None]).T @ (X * e[:, None])
    V = XtX_inv @ meat @ XtX_inv * n / (n - k)
    se = np.sqrt(np.diag(V)); p = 2 * stats.t.sf(np.abs(beta / se), n - k)
    return beta, se, p

R = df[(df["Savings_Balance"] > 0)].dropna(subset=["Age", "Customer_Tenure_Years", "Gender", "Urban_Rural"]).copy()
R["y"] = np.log(R["Savings_Balance"]); R["app"] = (R["Mobile_App_User"] == "Yes").astype(float)
base_cols = ["app"]
ctrl = pd.get_dummies(R[["Gender", "Urban_Rural"]], drop_first=True).astype(float)
ctrl["Age"] = R["Age"]; ctrl["Tenure"] = R["Customer_Tenure_Years"]
seg_d = pd.get_dummies(R["Customer_Segment"], prefix="Seg", drop_first=True).astype(float)

models = {"1. Unadjusted": R[["app"]], "2. + age, tenure, gender, urban/rural": pd.concat([R[["app"]], ctrl], axis=1),
          "3. + segment (bad control: see note)": pd.concat([R[["app"]], ctrl, seg_d], axis=1)}
out = []
for name, X in models.items():
    X = sm_X = np.column_stack([np.ones(len(X)), X.values]); beta, se, p = ols_hc1(R["y"], X)
    out.append({"Model": name, "n": len(R), "App effect on savings %": (np.exp(beta[1]) - 1) * 100,
                "95% CI low %": (np.exp(beta[1] - 1.96 * se[1]) - 1) * 100, "95% CI high %": (np.exp(beta[1] + 1.96 * se[1]) - 1) * 100, "p-value": p[1]})
reg = pd.DataFrame(out); display(reg.round(3))
print("Every confidence interval spans 0%: app use is not reliably associated with higher (or lower) savings, with or without controls.")

### Definition check and engagement intensity
`Mobile_App_User` contradicts other columns for some customers (customers with `Mobile_App_User = No` who still list the App as their preferred channel or have push notifications on). So test other plausible definitions of "digitally engaged", and check whether *more intense* app use (recent logins, more transactions) goes with higher savings.

In [ ]:
D = S.copy()
D["recency_days"] = (AS_OF - D["Last_Login_Date"]).dt.days
definitions = {
    "Registered app user (the flag)":            D["Mobile_App_User"] == "Yes",
    "Recently active (login within 90 days)":   (D["Mobile_App_User"] == "Yes") & (D["recency_days"] <= 90),
    "Preferred channel = App":                   D["Preferred_Channel"] == "App",
    "Push notifications enabled":                D["Push_Notification_Enabled"] == "Yes",
}
rows = []
for name, m in definitions.items():
    x, y = D.loc[m, "Savings_Balance"], D.loc[~m, "Savings_Balance"]
    rows.append({"Definition of 'digital'": name, "n_yes": len(x), "n_no": len(y), "median_yes": x.median(), "median_no": y.median(),
                 "median_gap_%": (x.median() / y.median() - 1) * 100, "MWU_p": stats.mannwhitneyu(x, y).pvalue})
display(pd.DataFrame(rows).round({"median_yes": 0, "median_no": 0, "median_gap_%": 1, "MWU_p": 3}))

au = D[D["Mobile_App_User"] == "Yes"].dropna(subset=["recency_days"])
r1 = stats.spearmanr(au["recency_days"], au["Savings_Balance"]); r2 = stats.spearmanr(au["Monthly_Transaction_Count"], au["Savings_Balance"], nan_policy="omit")
print(f"\nAmong app users: days since last login vs savings: rho = {r1.statistic:+.3f} (p = {r1.pvalue:.2f})")
print(f"Among app users: monthly transactions vs savings:  rho = {r2.statistic:+.3f} (p = {r2.pvalue:.2f})   (|rho| < 0.1 is negligible)")
print("Data contradictions behind the app flag:", int(((df["Preferred_Channel"] == "App") & (df["Mobile_App_User"] == "No")).sum()),
      "non-users prefer the App channel;", int(((df["Push_Notification_Enabled"] == "Yes") & (df["Mobile_App_User"] == "No")).sum()), "non-users have push notifications enabled.")
print("Note: median gaps are noisy for skewed data; rely on the p-values (rank test) and the CIs, not on the raw gap.")

In [ ]:
pval_u = u_res.pvalue
print("Q3A VERDICT")
print(f"  Statistical : NOT significant (Welch p={t_res.pvalue:.2f}, Mann-Whitney p={pval_u:.2f}); bootstrap CI for the mean gap spans zero.")
print(f"  Practical   : tiny. Medians differ by {np.median(a)/np.median(b)-1:+.1%}; means by {a.mean()/b.mean()-1:+.1%} (app users are slightly LOWER on the mean).")
print( "  Controlled  : no reliable difference in any regression (every 95% CI spans 0%), even after adjusting for age, tenure, gender and location.")
print(f"  Within segments: app users' medians lean higher ({seg_app['median_gap_%'].min():+.0f}% to {seg_app['median_gap_%'].max():+.0f}%) but none is significant: a hint worth re-testing on more data (esp. Premium), not a finding.")
print( "  Intensity   : login recency and transaction counts show no link to savings.")
print( "  Causal      : cannot be claimed either way. This is one snapshot, the app flag is internally inconsistent, and savings appear driven mainly by segment (which is itself assigned from balance).")
print("\n  => Answer: NO. App users do not hold meaningfully higher balances. Digital adoption looks like an engagement/cost-to-serve lever, not a savings-growth lever.")

---
# Q3B: Cross-sell opportunity: savings but no FD and no Insurance

**Target group (as asked):** customers with a **positive savings balance**, `Has_Fixed_Deposit == "No"` **and** `Has_Insurance == "No"`.
Customers whose savings balance was invalid/missing (cleaned to blank) cannot be valued and are counted separately, not silently dropped.

Three ways this analysis goes beyond a headline number:
1. the full product-holding grid (so the target group is seen in context),
2. **who is worth contacting first** (value concentration by segment),
3. **who can be contacted now** (account status, KYC and flags decide readiness).

In [ ]:
has_sav = df["Savings_Balance"] > 0
no_fd, no_ins = df["Has_Fixed_Deposit"] == "No", df["Has_Insurance"] == "No"
target = df[has_sav & no_fd & no_ins].copy()

n_t, v_t = len(target), target["Savings_Balance"].sum()
tot_cust, tot_val = int(has_sav.sum()), df.loc[has_sav, "Savings_Balance"].sum()
print(f">>> ANSWER: {n_t} customers hold savings but have NO Fixed Deposit and NO Insurance")
print(f">>> Total savings held by this group: LKR {v_t:,.0f}  (LKR {v_t/1e6:,.1f} million)")
print(f"    = {n_t/tot_cust:.1%} of customers with a valid balance and {v_t/tot_val:.1%} of all savings")
print(f"    Average per customer: LKR {v_t/n_t:,.0f}  | median: LKR {target['Savings_Balance'].median():,.0f}")
print(f"    Customers with no FD & no insurance but an invalid/missing balance (not valued): {int((no_fd & no_ins & ~has_sav).sum())}")

grid = df[has_sav].copy(); grid["FD"] = np.where(grid["Has_Fixed_Deposit"] == "Yes", "Has FD", "No FD"); grid["Insurance"] = np.where(grid["Has_Insurance"] == "Yes", "Has insurance", "No insurance")
print("\nProduct-holding grid: customers, then savings value (LKR m):")
display(pd.crosstab(grid["FD"], grid["Insurance"], margins=True))
display((pd.crosstab(grid["FD"], grid["Insurance"], values=grid["Savings_Balance"], aggfunc="sum", margins=True) / 1e6).round(1))

In [ ]:
by_seg = target.groupby("Customer_Segment")["Savings_Balance"].agg(customers="count", total="sum", mean="mean", median="median")
by_seg["% of group customers"] = by_seg["customers"] / n_t * 100
by_seg["% of group value"] = by_seg["total"] / v_t * 100
display(by_seg.round({"total": 0, "mean": 0, "median": 0, "% of group customers": 1, "% of group value": 1}))

srt = target["Savings_Balance"].sort_values(ascending=False)
print(f"Concentration: the top 10% of the group ({int(len(srt)*0.1)} customers) hold {srt.head(int(len(srt)*0.1)).sum()/v_t:.0%} of the group's savings.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(by_seg.index, by_seg["customers"], color="#1f4e79"); ax[0].set_title("Cross-sell group: customers by segment")
for i, v in enumerate(by_seg["customers"]): ax[0].text(i, v + 2, str(v), ha="center")
ax[1].bar(by_seg.index, by_seg["total"], color="#2e8b57"); ax[1].set_title("Cross-sell group: savings value by segment"); ax[1].yaxis.set_major_formatter(FuncFormatter(lkr_fmt))
for i, v in enumerate(by_seg["total"]): ax[1].text(i, v, lkr_fmt(v), ha="center", va="bottom")
plt.tight_layout(); plt.savefig(OUT / "q3b_crosssell_segments.png", dpi=150); plt.show()

### Who can actually be approached today? (readiness tiers)
A cross-sell list is only useful if the customers can be sold to. Compliance and account status decide that:
- **Tier A: ready now:** Active account, KYC *Verified*, no suspicious-activity flag, not a PEP.
- **Tier B: fix first:** Active, but KYC expired/pending/unknown, or flagged suspicious/PEP (needs a compliance step before any sale).
- **Tier C: reactivate first:** Inactive or Suspended account.

In [ ]:
comp_ok = (target["KYC_Status"] == "Verified") & (target["Suspicious_Activity_Flag"] != "Yes") & (target["PEP_Flag"] != "Yes")
active = target["Account_Status"] == "Active"
target["Tier"] = np.select([active & comp_ok, active & ~comp_ok],
                           ["A: Ready now", "B: Fix compliance first"], default="C: Reactivate first")
tier = target.groupby("Tier")["Savings_Balance"].agg(customers="count", total="sum")
tier["% of group value"] = tier["total"] / v_t * 100
display(tier.round({"total": 0, "% of group value": 1}))
tier_seg = target.pivot_table(index="Tier", columns="Customer_Segment", values="Savings_Balance", aggfunc=["count", "sum"]).fillna(0)
display(tier_seg.round(0))

A = target[target["Tier"] == "A: Ready now"]
print(f"PRIORITY LIST (Tier A): {len(A)} customers holding LKR {A['Savings_Balance'].sum()/1e6:,.1f}m")
print(f"  of which Premium: {int((A['Customer_Segment']=='Premium').sum())} customers holding LKR {A.loc[A['Customer_Segment']=='Premium','Savings_Balance'].sum()/1e6:,.1f}m")

In [ ]:
# Caution overlay: do NOT push extra products to customers who are already stretched on a loan
LD2 = df[df["Loan_Data_Complete"] & (df["Savings_Balance"] > 0)].copy(); LD2["DSR"] = LD2["Outstanding_Loan_Balance"] / LD2["Savings_Balance"]
stretched = set(LD2.loc[LD2["DSR"] > 1, "Customer_ID"])
A_safe = A[~A["Customer_ID"].isin(stretched)]
tgt_stretched = target[target["Customer_ID"].isin(stretched)]
print(f"Cross-sell group members who also hold a loan with Debt-to-Savings > 1: {len(tgt_stretched)} customers (LKR {tgt_stretched['Savings_Balance'].sum()/1e6:,.1f}m of savings)")
print(f"Tier A after removing stretched borrowers: {len(A_safe)} customers, LKR {A_safe['Savings_Balance'].sum()/1e6:,.1f}m")

# Illustrative scenario: the assumptions below are NOT from the data. Replace with real campaign results.
CONVERSION = [0.05, 0.10, 0.20]      # % of Tier-A customers who take up an FD / insurance-linked product
SHARE_MOVED = 0.30                   # share of their savings moved into the new product
scen = pd.DataFrame({"Take-up rate": [f"{c:.0%}" for c in CONVERSION],
                     "Customers": [round(len(A_safe) * c) for c in CONVERSION],
                     "Savings that could migrate (LKR m)": [round(A_safe["Savings_Balance"].sum() * c * SHARE_MOVED / 1e6, 1) for c in CONVERSION]})
print(f"\nILLUSTRATIVE scenario (assumes {SHARE_MOVED:.0%} of balance moves per taker; assumptions are NOT from the dataset):")
display(scen)

In [ ]:
print("Q3B SUMMARY")
print(f"  {n_t} customers ({n_t/tot_cust:.0%} of those with savings) hold LKR {v_t/1e6:,.1f}m ({v_t/tot_val:.0%} of all savings) with no FD and no insurance.")
print(f"  Value is concentrated: Premium = {by_seg.loc['Premium','customers']} customers but {by_seg.loc['Premium','% of group value']:.0f}% of the group's value; Starter = {by_seg.loc['Starter','customers']} customers but only {by_seg.loc['Starter','% of group value']:.0f}%.")
print(f"  Contactable now (Tier A, not stretched): {len(A_safe)} customers, LKR {A_safe['Savings_Balance'].sum()/1e6:,.1f}m. The rest needs KYC/compliance or reactivation first.")

---
# Q3C: Which acquisition channel *consistently* produces the best customers?

The question needs **both** conditions: the **highest savings** *and* the **lowest default rate**, *consistently*. Therefore:
1. a channel scorecard on several metrics (not just one),
2. ranks across metrics,
3. a **bootstrap stability test**: resample customers thousands of times and ask *how often* each channel comes out on top. A channel that is best in 90% of resamples is consistent; one that is best in 30% is a lucky draw.

In [ ]:
L = df[df["Loan_Data_Complete"]].copy()
L["is_def"] = L["Loan_Repayment_Status"] == "Defaulted"; L["is_prob"] = L["Loan_Repayment_Status"].isin(["Defaulted", "Delayed"])
chs = sorted(df["Acquisition_Channel"].dropna().unique())

g = df.groupby("Acquisition_Channel")
ch = g["Savings_Balance"].agg(customers="size", valid_balances="count", mean_savings="mean", median_savings="median", total_savings="sum")
gl = L.groupby("Acquisition_Channel")
ch["loans"] = gl.size(); ch["defaults"] = gl["is_def"].sum()
ch["default_rate_%"] = ch["defaults"] / ch["loans"] * 100
ci = [stats.binomtest(int(d), int(n)).proportion_ci(method="wilson") for d, n in zip(ch["defaults"], ch["loans"])]
ch["def_ci_low_%"] = [c.low * 100 for c in ci]; ch["def_ci_high_%"] = [c.high * 100 for c in ci]
ch["problem_rate_%"] = gl["is_prob"].mean() * 100
ch["loan_penetration_%"] = g["Loan_Holder"].mean() * 100
ch["loan_book_share_%"] = g["Outstanding_Loan_Balance"].sum() / df["Outstanding_Loan_Balance"].sum() * 100
ch["active_accounts_%"] = g["Account_Status"].apply(lambda s: (s == "Active").mean() * 100)
ch["kyc_verified_%"] = g["KYC_Status"].apply(lambda s: (s == "Verified").mean() * 100)
ch["FD_holders_%"] = g["Has_Fixed_Deposit"].apply(lambda s: (s == "Yes").mean() * 100)
ch["insured_%"] = g["Has_Insurance"].apply(lambda s: (s == "Yes").mean() * 100)
display(ch.round({"mean_savings": 0, "median_savings": 0, "total_savings": 0, "default_rate_%": 1, "def_ci_low_%": 1, "def_ci_high_%": 1,
                  "problem_rate_%": 1, "loan_penetration_%": 1, "loan_book_share_%": 1, "active_accounts_%": 1, "kyc_verified_%": 1, "FD_holders_%": 1, "insured_%": 1}).T)

p_sav = stats.kruskal(*[x["Savings_Balance"].dropna() for _, x in df.groupby("Acquisition_Channel")]).pvalue
p_def_ch = stats.chi2_contingency(pd.crosstab(L["Acquisition_Channel"], L["is_def"]))[1]
print(f"\nAre the channel differences real?  Savings (Kruskal-Wallis) p = {p_sav:.2f}   |   Default rate (chi-square) p = {p_def_ch:.2f}")
print("Segment mix by channel (%): customers are similarly mixed, so segment is not driving the differences:")
display((pd.crosstab(df["Acquisition_Channel"], df["Customer_Segment"], normalize="index") * 100).round(1))

In [ ]:
# ---- rank scorecard (1 = best) ----------------------------------------------------------------------------
score = pd.DataFrame({
    "Mean savings (high=good)":   ch["mean_savings"].rank(ascending=False),
    "Median savings (high=good)": ch["median_savings"].rank(ascending=False),
    "Default rate (low=good)":    ch["default_rate_%"].rank(ascending=True),
    "Problem-loan rate (low=good)": ch["problem_rate_%"].rank(ascending=True),
    "Active accounts (high=good)": ch["active_accounts_%"].rank(ascending=False),
    "KYC verified (high=good)":   ch["kyc_verified_%"].rank(ascending=False),
})
score["Average rank"] = score.mean(axis=1)
display(score.sort_values("Average rank"))
core = score.iloc[:, :4].copy(); core["Core avg rank (savings + risk)"] = core.mean(axis=1)
print("Ranks 1-4. A channel that is truly 'consistently best' would show rank 1 across the savings AND risk columns.")
display(core.sort_values("Core avg rank (savings + risk)"))

### Stability test: how often does each channel win on resampling?
Resample customers *within each channel* with replacement (thousands of times), recompute the channel metrics, and record who is best. Then compute the probability that **one and the same channel** is simultaneously best on savings *and* lowest on defaults.

In [ ]:
sav = {c: df.loc[(df["Acquisition_Channel"] == c) & df["Savings_Balance"].notna(), "Savings_Balance"].values for c in chs}
dfl = {c: L.loc[L["Acquisition_Channel"] == c, "is_def"].values.astype(float) for c in chs}
med = np.empty((N_BOOT, len(chs))); mean_ = np.empty_like(med); dr = np.empty_like(med)
for i in range(N_BOOT):
    for j, c in enumerate(chs):
        s = rng.choice(sav[c], len(sav[c])); med[i, j] = np.median(s); mean_[i, j] = s.mean()
        dr[i, j] = rng.choice(dfl[c], len(dfl[c])).mean()
dr_tb = dr + rng.random(dr.shape) * 1e-9                         # random tie-break for equal default rates
best_med, best_mean, best_def = med.argmax(1), mean_.argmax(1), dr_tb.argmin(1)

stab = pd.DataFrame({
    "P(highest MEDIAN savings)": [(best_med == j).mean() for j in range(len(chs))],
    "P(highest MEAN savings)":   [(best_mean == j).mean() for j in range(len(chs))],
    "P(lowest default rate)":    [(best_def == j).mean() for j in range(len(chs))],
    "P(best median AND lowest default)": [((best_med == j) & (best_def == j)).mean() for j in range(len(chs))],
    "P(best mean AND lowest default)":   [((best_mean == j) & (best_def == j)).mean() for j in range(len(chs))],
}, index=chs) * 100
display(stab.round(1))
print(f"Probability that a single channel is best on BOTH median savings and default rate: {(best_med == best_def).mean():.0%}")
print(f"Probability that a single channel is best on BOTH mean savings and default rate:   {(best_mean == best_def).mean():.0%}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
x = np.arange(len(chs)); w = 0.38
ax[0].bar(x - w/2, ch["mean_savings"], w, label="Mean", color="#1f4e79"); ax[0].bar(x + w/2, ch["median_savings"], w, label="Median", color="#2e8b57")
ax[0].set_xticks(x); ax[0].set_xticklabels(chs); ax[0].yaxis.set_major_formatter(FuncFormatter(lkr_fmt)); ax[0].legend(); ax[0].set_title("Savings per customer by channel")
ax[1].bar(chs, ch["default_rate_%"], color="#c0392b")
ax[1].errorbar(chs, ch["default_rate_%"], yerr=[ch["default_rate_%"] - ch["def_ci_low_%"], ch["def_ci_high_%"] - ch["default_rate_%"]], fmt="none", ecolor="black", capsize=4)
ax[1].set_title("Default rate by channel (whiskers = 95% CI)"); ax[1].set_ylabel("%")
ax[2].scatter(ch["median_savings"], ch["default_rate_%"], s=ch["customers"] * 4, color="#1f4e79", alpha=.7)
for c_, r in ch.iterrows(): ax[2].annotate(c_, (r["median_savings"], r["default_rate_%"]), textcoords="offset points", xytext=(8, 6))
ax[2].set_xlabel("Median savings (LKR): higher is better"); ax[2].set_ylabel("Default rate %: lower is better")
ax[2].xaxis.set_major_formatter(FuncFormatter(lkr_fmt)); ax[2].set_title("Ideal channel = bottom-right corner")
plt.tight_layout(); plt.savefig(OUT / "q3c_channels.png", dpi=150); plt.show()

In [ ]:
best_def_ch, best_med_ch, best_mean_ch = ch["default_rate_%"].idxmin(), ch["median_savings"].idxmax(), ch["mean_savings"].idxmax()
print("Q3C FINDINGS")
print(f"  Lowest default rate : {best_def_ch} ({ch.loc[best_def_ch,'default_rate_%']:.1f}%)   |   Highest MEDIAN savings: {best_med_ch} (LKR {ch.loc[best_med_ch,'median_savings']:,.0f})   |   Highest MEAN savings: {best_mean_ch} (LKR {ch.loc[best_mean_ch,'mean_savings']:,.0f})")
print(f"  Same channel on both? {'YES' if best_def_ch in (best_med_ch, best_mean_ch) else 'NO'}.")
print(f"  {best_def_ch} is the safest, but ranks {int(score.loc[best_def_ch,'Mean savings (high=good)'])} of 4 on mean savings and {int(score.loc[best_def_ch,'Median savings (high=good)'])} of 4 on median savings; "
      f"{best_mean_ch} has the highest mean balance but {ch.loc[best_mean_ch,'default_rate_%']:.1f}% defaults.")
print(f"  Are any differences statistically reliable?  Savings p = {p_sav:.2f}; default rate p = {p_def_ch:.2f}  -> {'yes' if min(p_sav, p_def_ch) < .05 else 'no'}.")
print(f"  Probability one channel wins both on resampling: {(best_med == best_def).mean():.0%} (median) / {(best_mean == best_def).mean():.0%} (mean).")
print(f"  Stability: {best_def_ch} shows the lowest default rate in {(best_def == chs.index(best_def_ch)).mean():.0%} of resamples (the most dependable risk result), while the top-savings spot is shared between "
      + ", ".join(f"{c} ({stab.loc[c,'P(highest MEDIAN savings)']:.0f}%)" for c in stab['P(highest MEDIAN savings)'].sort_values(ascending=False).index[:3]) + " (median basis).")
print(f"  Lending lens: {ch['loan_penetration_%'].idxmax()} has the highest loan penetration ({ch['loan_penetration_%'].max():.0f}%, not statistically distinguishable) and {ch['loan_book_share_%'].idxmax()} the largest share of the outstanding loan book ({ch['loan_book_share_%'].max():.0f}%).")
print(f"  Best all-rounder on average rank (savings + risk): {core['Core avg rank (savings + risk)'].idxmin()} (but it is never #1 on both, and its account-activity/KYC quality is the weakest).")

**What this implies for the business:**
- **No channel is "consistently" best on both measures.** On resampling, a single channel wins both savings *and* default rate only about 1 time in 10. So the evidence does not justify moving the whole acquisition budget to one channel.
- The channels have **different strengths**: **Agent** is the most dependably *low-default* channel (lowest default rate in roughly 4 of 5 resamples, although the overall gap is not statistically significant) and has the highest loan penetration, but its customers hold the smallest average balances. **Digital** has the best *typical* (median) balance and is the best all-rounder on average rank. **Branch** has the highest *mean* balance. **Referral** has the highest delinquency.
- That points to a **portfolio strategy**: use Agents for lending volume, Digital/Branch for savings mobilisation, and match the product push to the channel, instead of a "winner takes all" decision.
- Because the differences are small relative to the sample, the practical recommendation is to **track channel quality monthly** (a KPI for Q4), **run a controlled test** before reallocating spend, and **tighten credit checks on Referral/Branch/Digital lending** until the default gap is measured on more loans.
- Look beyond the headline: **Digital has the lowest KYC-verified share (about 66%) and the lowest share of active accounts (about 71%)**, a quality issue to fix whatever its balances look like.

---
## Export

In [ ]:
tables = {
    "Cleaning_Log": pd.DataFrame(cleaning_log),
    "Q3A_AppVsNon": grp, "Q3A_Tests": pd.DataFrame({"Test": ["Welch t p", "Mann-Whitney p", "Cliff delta", "Mean gap CI low", "Mean gap CI high"],
                                                     "Value": [t_res.pvalue, u_res.pvalue, cliff, ci_mean[0], ci_mean[1]]}),
    "Q3A_BySegment": seg_app, "Q3A_Regression": reg,
    "Q3B_Grid_Counts": pd.crosstab(grid["FD"], grid["Insurance"], margins=True), "Q3B_BySegment": by_seg, "Q3B_Tiers": tier,
    "Q3B_Scenario": scen, "Q3B_TierA_List": A_safe[["Customer_ID", "Customer_Segment", "Savings_Balance", "Province", "Account_Status", "KYC_Status"]].sort_values("Savings_Balance", ascending=False),
    "Q3C_Channels": ch.T, "Q3C_Scorecard": score, "Q3C_Stability": stab,
}
with pd.ExcelWriter(OUT / "Q3_Workings.xlsx", engine="openpyxl") as xw:
    for name, t in tables.items():
        t.round(3).to_excel(xw, sheet_name=name[:31], index=not isinstance(t.index, pd.RangeIndex))
    for ws in xw.book.worksheets:
        ws.freeze_panes = "A2"
        for colx in ws.columns:
            w = max(len(str(c.value)) if c.value is not None else 0 for c in colx[:40])
            ws.column_dimensions[colx[0].column_letter].width = min(max(w + 2, 10), 50)
print("Saved to", OUT.resolve(), ":", sorted(p.name for p in OUT.iterdir()))

---
# Q4. Five KPIs and dashboard

### How the five KPIs were chosen
A good KPI is **(1)** calculable monthly from fields already held, **(2)** tied to a *decision*, **(3)** backed by a finding in Q1-Q3, and **(4)** balanced: together the five cover different areas of customer health.

| # | Area | KPI | Finding that motivates it |
|---|---|---|---|
| K1 | **Risk** | Loan Delinquency Rate | Only ~6 in 10 loans are fully on time (Q2) |
| K2 | **Compliance** | KYC Verified Rate | ~30% of customers lack verified KYC; some are PEP / suspicious-flagged |
| K3 | **Engagement / retention** | Active Account Rate | Inactive/suspended accounts hold ~29% of savings |
| K4 | **Growth** | Cross-sell Coverage Rate | LKR 222M sits with customers with no FD and no insurance (Q3B) |
| K5 | **Savings health (early warning)** | Fragile Saver Rate | Every customer has positive net flow (Q1C), so the real signal is a *thin cushion* |

**Monthly use:** all KPIs come from one function, `compute_kpis(df)`. Each month, load that month's extract, apply the same cleaning (Section 0), call the function and append the result (Q4.7).

In [ ]:
OUT = OUT_ROOT / "q4"; OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

---
## Q4.1 KPI engine
One function computes every KPI **and** the context numbers behind it (numerator, denominator, 95% confidence interval). Rates built on small samples (K1 has only 218 loans) must show their uncertainty, so a monthly wobble is not mistaken for a trend.

In [ ]:
def wilson(k, n):
    lo, hi = stats.binomtest(int(k), int(n)).proportion_ci(method="wilson")
    return lo * 100, hi * 100

def compute_kpis(d):
    """Return a tidy KPI table for one monthly snapshot of cleaned data `d`."""
    rows = []
    # K1 RISK: Loan Delinquency Rate = loans Delayed or Defaulted / loans with complete data
    L_ = d[d["Loan_Data_Complete"]]
    num = int(L_["Loan_Repayment_Status"].isin(["Delayed", "Defaulted"]).sum()); den = len(L_)
    rows.append(("K1", "Risk", "Loan Delinquency Rate", num, den))
    # K2 COMPLIANCE: KYC Verified Rate = customers with KYC_Status 'Verified' / all customers
    rows.append(("K2", "Compliance", "KYC Verified Rate", int((d["KYC_Status"] == "Verified").sum()), len(d)))
    # K3 ENGAGEMENT: Active Account Rate = Account_Status 'Active' / all customers
    rows.append(("K3", "Engagement / retention", "Active Account Rate", int((d["Account_Status"] == "Active").sum()), len(d)))
    # K4 GROWTH: Cross-sell Coverage = customers (valid savings) holding an FD or Insurance / customers with valid savings
    v = d[d["Savings_Balance"] > 0]
    cov = int(((v["Has_Fixed_Deposit"] == "Yes") | (v["Has_Insurance"] == "Yes")).sum())
    rows.append(("K4", "Growth", "Cross-sell Coverage Rate", cov, len(v)))
    # K5 SAVINGS HEALTH: Fragile Saver Rate = customers withdrawing > 80% of monthly deposits / customers with valid flow data
    wr = d["Monthly_Withdrawal_Avg"] / d["Monthly_Deposit_Avg"]
    ok = wr.notna()
    rows.append(("K5", "Savings health", "Fragile Saver Rate", int((wr[ok] > FRAGILE_THRESHOLD).sum()), int(ok.sum())))
    t = pd.DataFrame(rows, columns=["id", "area", "kpi", "numerator", "denominator"])
    t["value_%"] = t["numerator"] / t["denominator"] * 100
    ci = [wilson(k, n) for k, n in zip(t["numerator"], t["denominator"])]
    t["ci_low_%"] = [c[0] for c in ci]; t["ci_high_%"] = [c[1] for c in ci]
    return t

FRAGILE_THRESHOLD = 0.80          # a customer is 'fragile' if withdrawals exceed 80% of deposits (sensitivity in Q4.4)
kpi_now = compute_kpis(df)
display(kpi_now.round(1))

### Q4.1a Reconciliation with Q1-Q3
A KPI that does not tie back to Q1-Q3 will be challenged. These checks make sure the numbers agree.

In [ ]:
Lc = df[df["Loan_Data_Complete"]]
n_def, n_loans_c = int((Lc["Loan_Repayment_Status"] == "Defaulted").sum()), len(Lc)
assert (n_def, n_loans_c) == (22, 218), "default count should match Q2"
v_ = df[df["Savings_Balance"] > 0]
q3_target = int(((v_["Has_Fixed_Deposit"] == "No") & (v_["Has_Insurance"] == "No")).sum())
assert q3_target == 251, "no-FD-no-insurance customers should match Q3B"
k4 = kpi_now.set_index("id").loc["K4"]; assert k4["denominator"] - k4["numerator"] == q3_target
print("Reconciled: 22 defaults / 218 loans (Q2); 251 customers with no FD and no insurance (Q3B) = K4 denominator minus numerator.")

# supporting numbers shown next to the KPIs
default_rate = n_def / n_loans_c * 100
hi_risk = (df["PEP_Flag"] == "Yes") | (df["Suspicious_Activity_Flag"] == "Yes")
hi_risk_unverified = int((hi_risk & (df["KYC_Status"] != "Verified")).sum())
nonactive_sav_share = df.loc[df["Account_Status"] != "Active", "Savings_Balance"].sum() / df["Savings_Balance"].sum() * 100
nonactive_sav_lkr = df.loc[df["Account_Status"] != "Active", "Savings_Balance"].sum()
nocross = v_[(v_["Has_Fixed_Deposit"] == "No") & (v_["Has_Insurance"] == "No")]
print(f"Supporting: default rate {default_rate:.1f}% | PEP/suspicious customers without verified KYC: {hi_risk_unverified} | "
      f"savings in non-active accounts: LKR {nonactive_sav_lkr/1e6:,.0f}m ({nonactive_sav_share:.0f}%) | cross-sell pool: LKR {nocross['Savings_Balance'].sum()/1e6:,.1f}m")

---
## Q4.2 Targets, RAG status and the reasoning behind each target
**Target principle:** a target must be *achievable* and *justified*. Wherever possible a target is anchored on a level **the business already achieves in its own best-performing segment**, which is more credible than a number from nowhere. Regulatory KPIs use the regulatory expectation instead.

**RAG rule:** Green = target met. Amber = within 20% of the target. Red = more than 20% away.

In [ ]:
meta = pd.DataFrame([
 dict(id="K1", direction="lower", target=32.0,
      definition="Share of loans whose repayment status is Delayed or Defaulted.",
      formula="COUNT(loans with Loan_Repayment_Status IN {Delayed, Defaulted}) / COUNT(loans with complete loan data)",
      why="The early-warning signal for credit losses. Defaults are rare (10%), but delinquency is large enough to act on and moves before defaults do.",
      decision="Tighten underwriting by channel/product, size the collections team, set provisioning and lending limits.",
      target_logic="Agent-acquired loans run at ~32%, so 32% is a STRETCH target (Agent's figure rests on 56 loans, 95% CI 21-45%, so it may be partly luck: validate with a 6-month pilot). Stretch 25% in 24 months. Guard-rail: default rate <= 6% (Agent 5.4%, Business loans 6.8% today). Add the latest CBSL finance-company NPL ratio as an external reference (note: 'Delayed' is broader than a formal NPL).",
      owner="Chief Risk Officer", trigger="Red for 2 consecutive months -> credit-policy review"),
 dict(id="K2", direction="higher", target=98.0,
      definition="Share of customers whose KYC status is Verified (not Expired, Pending or Unknown).",
      formula="COUNT(KYC_Status = 'Verified') / COUNT(all customers)",
      why="KYC / customer-due-diligence is a regulatory obligation for licensed finance companies, so this is a licence-protection KPI, not a nice-to-have.",
      decision="Staff the KYC remediation team, restrict transactions on expired KYC, prioritise PEP/suspicious customers first.",
      target_logic="The regulatory floor is 100%; 98% allows for customers mid-renewal and new onboarding. Reaching it means remediating ~139 customers (expired + pending). Interim milestone 85% at 6 months. PEP/suspicious customers without verified KYC must reach ZERO within 30 days.",
      owner="Head of Compliance", trigger="Any PEP/suspicious customer unverified -> immediate escalation"),
 dict(id="K3", direction="higher", target=82.0,
      definition="Share of customer accounts that are Active (not Inactive or Suspended).",
      formula="COUNT(Account_Status = 'Active') / COUNT(all customers)",
      why="Dormant accounts earn nothing, cost money to serve, and are the first step to attrition. Retention is cheaper than acquisition.",
      decision="Fund reactivation campaigns, target outreach by province/channel, decide where to invest in service.",
      target_logic="Southern province (81%) and the Branch channel (80%) already show it is achievable; 82% means lifting every cohort to today's best (small cohorts, indicative). Caveat: status codes track behaviour only loosely (115 of 123 non-active accounts still transact), so treat K3 as a first-pass measure. About 29% of savings sits in non-active accounts.",
      owner="Head of Customer Operations", trigger="Fall of >2 pts in a month -> root-cause review"),
 dict(id="K4", direction="higher", target=55.0,
      definition="Share of customers with savings who also hold a Fixed Deposit or Insurance.",
      formula="COUNT(valid savings AND (Has_Fixed_Deposit='Yes' OR Has_Insurance='Yes')) / COUNT(valid savings)",
      why="The main growth lever that needs no new customers: deeper relationships earn more and churn less. Q3B found LKR 222M sitting with customers who hold neither product.",
      decision="Allocate campaign budget and relationship-manager time, prioritising Premium (lowest coverage, highest value).",
      target_logic="55% in 12 months needs ~31 more covered customers, i.e. converting about one in three of the 95 customers who are contactable today and not over-leveraged (Tier A, Q3B). Longer-term 65%.",
      owner="Head of Retail Products", trigger="Below plan for 2 months -> change campaign mix"),
 dict(id="K5", direction="lower", target=12.0,
      definition="Share of customers who withdraw more than 80% of what they deposit each month.",
      formula="COUNT(Monthly_Withdrawal_Avg / Monthly_Deposit_Avg > 0.80) / COUNT(customers with valid deposit & withdrawal data)",
      why="Q1C showed nobody is formally 'draining', so the real signal is a thin cushion: these customers are one bad month from draining and are the least able to absorb a loan payment.",
      decision="Trigger financial-wellness nudges, pause credit-limit increases, and flag the segment/channel producing fragile savers.",
      target_logic="Digital (10%) and Premium (11%) already run near 10-11%; 12% asks Starter (23%) and Branch (28%) to move toward the best. Threshold sensitivity is tested in Q4.4.",
      owner="Head of Retail Banking", trigger="Rising 3 months in a row -> early-warning outreach"),
])
kt = kpi_now.merge(meta, on="id")
kt["gap_pts"] = np.where(kt["direction"] == "lower", kt["value_%"] - kt["target"], kt["target"] - kt["value_%"])   # positive = still short of target
kt["gap_ratio"] = kt["gap_pts"] / kt["target"]
kt["RAG"] = np.where(kt["gap_pts"] <= 0, "Green", np.where(kt["gap_ratio"] <= 0.20, "Amber", "Red"))
kt["target_txt"] = np.where(kt["direction"] == "lower", "≤ ", "≥ ") + kt["target"].round(0).astype(int).astype(str) + "%"
display(kt[["id", "area", "kpi", "numerator", "denominator", "value_%", "ci_low_%", "ci_high_%", "target_txt", "gap_pts", "RAG"]].round(1))

In [ ]:
# numbers quoted in the target logic: verify rather than trust
need_kyc = int(np.ceil(0.98 * len(df))) - int((df["KYC_Status"] == "Verified").sum())
k4r = kt.set_index("id").loc["K4"]; need_cov = int(np.ceil(0.55 * k4r["denominator"])) - int(k4r["numerator"])

# Tier A (Q3B): active + KYC verified + no suspicious/PEP flag, with no FD and no insurance, and not a stretched borrower (DSR>1)
tg = df[(df["Savings_Balance"] > 0) & (df["Has_Fixed_Deposit"] == "No") & (df["Has_Insurance"] == "No")]
tierA = tg[(tg["Account_Status"] == "Active") & (tg["KYC_Status"] == "Verified") & (tg["Suspicious_Activity_Flag"] != "Yes") & (tg["PEP_Flag"] != "Yes")]
LD2 = df[df["Loan_Data_Complete"] & (df["Savings_Balance"] > 0)]; stretched = set(LD2.loc[LD2["Outstanding_Loan_Balance"] / LD2["Savings_Balance"] > 1, "Customer_ID"])
tierA_safe = tierA[~tierA["Customer_ID"].isin(stretched)]
print(f"K2: customers to remediate to reach 98% KYC verified: {need_kyc}")
print(f"K4: extra covered customers needed for 55%: {need_cov}  =  {need_cov/len(tierA_safe):.0%} of the {len(tierA_safe)} contactable, non-stretched customers (Tier A)")
assert need_kyc == 139 and len(tierA_safe) == 95

---
## Q4.3 Where each KPI is worst (the breakdowns that make it actionable)
A headline number says *how bad*; a breakdown says *where to act*. These tables feed the dashboard charts. With small groups, treat differences as indicative.

In [ ]:
def rate_by(frame, col, flag_fn, den_fn=None):
    g = frame.groupby(col)
    num = g.apply(lambda x: int(flag_fn(x).sum()), include_groups=False)
    den = g.apply(lambda x: int(den_fn(x).sum()) if den_fn else len(x), include_groups=False)
    out = pd.DataFrame({"numerator": num, "denominator": den}); out["rate_%"] = out["numerator"] / out["denominator"] * 100
    return out

L = df[df["Loan_Data_Complete"]]
b1 = rate_by(L, "Acquisition_Channel", lambda x: x["Loan_Repayment_Status"].isin(["Delayed", "Defaulted"]))
b1["default_rate_%"] = L.groupby("Acquisition_Channel").apply(lambda x: (x["Loan_Repayment_Status"] == "Defaulted").mean() * 100, include_groups=False)
b2 = pd.crosstab(df["Acquisition_Channel"], df["KYC_Status"], normalize="index").mul(100)[["Verified", "Expired", "Pending", "Unknown"]]
b3 = rate_by(df, "Province", lambda x: x["Account_Status"] == "Active").sort_values("rate_%")
v = df[df["Savings_Balance"] > 0]
b4 = rate_by(v, "Customer_Segment", lambda x: (x["Has_Fixed_Deposit"] == "Yes") | (x["Has_Insurance"] == "Yes"))
b4["no_cover_savings_LKRm"] = v[(v["Has_Fixed_Deposit"] == "No") & (v["Has_Insurance"] == "No")].groupby("Customer_Segment")["Savings_Balance"].sum() / 1e6
wr_all = df["Monthly_Withdrawal_Avg"] / df["Monthly_Deposit_Avg"]
fl = df[wr_all.notna()].copy(); fl["fragile"] = (wr_all[wr_all.notna()] > FRAGILE_THRESHOLD)
b5 = rate_by(fl, "Acquisition_Channel", lambda x: x["fragile"])

for name, t in [("K1 delinquency by channel", b1), ("K2 KYC mix by channel (%)", b2), ("K3 active rate by province", b3),
                ("K4 coverage by segment", b4), ("K5 fragile savers by channel", b5)]:
    print(f"\n{name}"); display(t.round(1))

In [ ]:
# Are the 'worst cohort' differences statistically real? (so the dashboard does not over-claim)
def chi_p(frame, col, flag):
    return stats.chi2_contingency(pd.crosstab(frame[col], flag))[1]
def fp(p): return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
print(f"K1 delinquency vs channel:  chi-square p = {chi_p(L, 'Acquisition_Channel', L['Loan_Repayment_Status'].isin(['Delayed','Defaulted'])):.2f}")
print(f"K3 active rate vs province: chi-square p = {chi_p(df, 'Province', df['Account_Status'] == 'Active'):.2f}")
print(f"K4 coverage vs segment:     chi-square {fp(chi_p(v, 'Customer_Segment', (v['Has_Fixed_Deposit']=='Yes') | (v['Has_Insurance']=='Yes')))}")
print(f"K5 fragile vs channel:      chi-square {fp(chi_p(fl, 'Acquisition_Channel', fl['fragile']))}")
print(f"K5 fragile vs segment:      chi-square {fp(chi_p(fl, 'Customer_Segment', fl['fragile']))}")
print("\nInterpretation: the HEADLINE KPI levels are solid. Cohort differences for K1 (channel) and K3 (province) are indicative only, while K4 (Premium has lower cover) and K5 (Branch has more fragile savers) are statistically reliable.")

---
## Q4.4 Sensitivity: do the KPI values depend on the choices made?
The K5 threshold (80%) and the K1 definition (delayed *and* defaulted) are judgement calls. Show the value under alternatives so the choice can be defended.

In [ ]:
rows = []
for th in [0.70, 0.80, 0.90]:
    ok = wr_all.notna(); rows.append({"KPI": "K5 Fragile Saver Rate", "Variant": f"withdrawals > {th:.0%} of deposits", "Value %": round((wr_all[ok] > th).mean() * 100, 1)})
Ls = L["Loan_Repayment_Status"]
rows += [{"KPI": "K1 Delinquency", "Variant": "Delayed + Defaulted (headline)", "Value %": round(Ls.isin(["Delayed", "Defaulted"]).mean() * 100, 1)},
         {"KPI": "K1 Delinquency", "Variant": "Defaulted only", "Value %": round((Ls == "Defaulted").mean() * 100, 1)},
         {"KPI": "K1 Delinquency", "Variant": "Exposure-weighted (by outstanding balance)",
          "Value %": round(L.loc[Ls.isin(["Delayed", "Defaulted"]), "Outstanding_Loan_Balance"].sum() / L["Outstanding_Loan_Balance"].sum() * 100, 1)},
         {"KPI": "K2 KYC Verified", "Variant": "Unknown status excluded from denominator", "Value %": round((df["KYC_Status"] == "Verified").sum() / (df["KYC_Status"] != "Unknown").sum() * 100, 1)},
         {"KPI": "K3 Active", "Variant": "Active AND >=1 transaction/month", "Value %": round(((df["Account_Status"] == "Active") & (df["Monthly_Transaction_Count"] > 0)).mean() * 100, 1)},
         {"KPI": "K4 Coverage", "Variant": "Insurance only", "Value %": round((v["Has_Insurance"] == "Yes").mean() * 100, 1)},
         {"KPI": "K4 Coverage", "Variant": "Fixed deposit only", "Value %": round((v["Has_Fixed_Deposit"] == "Yes").mean() * 100, 1)}]
sens = pd.DataFrame(rows); display(sens)
print("Reading: K1/K2/K3/K4 are stable to reasonable alternative definitions. K5 is threshold-sensitive: it doubles at a 70% cut-off, and at 90% nobody qualifies (withdrawal ratios top out near 0.9). 80% keeps the KPI informative, but its level is less important than its TREND, which is why the dashboard states the threshold explicitly.")

---
## Q4.5 Dashboard
**Audience:** a non-technical business reader. Design rules used: one-glance answer per KPI (value, target, traffic light), one chart per KPI showing *where* the problem is, an "actions" strip with plain-language takeaways, and a small-print footer on data caveats.

**Page 1** = the dashboard. **Page 2** = a one-page KPI dictionary (definition, formula, target logic).

In [ ]:
import textwrap
from matplotlib.patches import Rectangle
from matplotlib.backends.backend_pdf import PdfPages

GREEN, AMBER, RED, NAVY, GREY, BG = "#2e8b57", "#e0a800", "#c0392b", "#1f4e79", "#6b7280", "#f4f6f9"
RAGC = {"Green": GREEN, "Amber": AMBER, "Red": RED}
K = kt.set_index("id")

sub_text = {
    "K1": f"{int(K.loc['K1','numerator'])} of {int(K.loc['K1','denominator'])} loans delayed or defaulted  |  default rate {default_rate:.1f}%",
    "K2": f"{int(K.loc['K2','numerator'])} of {int(K.loc['K2','denominator'])} verified  |  {hi_risk_unverified} PEP/suspicious not verified",
    "K3": f"{int(K.loc['K3','numerator'])} of {int(K.loc['K3','denominator'])} active  |  {nonactive_sav_share:.0f}% of savings in non-active accounts",
    "K4": f"{int(K.loc['K4','numerator'])} of {int(K.loc['K4','denominator'])} hold FD or insurance  |  LKR {nocross['Savings_Balance'].sum()/1e6:,.0f}m uncovered",
    "K5": f"{int(K.loc['K5','numerator'])} of {int(K.loc['K5','denominator'])} customers withdraw >80% of deposits",
}
chart_title = {"K1": "Delinquency rate by acquisition channel", "K2": "KYC status mix by channel (%)", "K3": "Active account rate by province",
               "K4": "Cross-sell coverage by segment", "K5": "Fragile-saver rate by channel"}

def tile(ax, k):
    r = K.loc[k]; col = RAGC[r["RAG"]]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(Rectangle((0, 0), 1, 1, fc="white", ec="#d0d7de", lw=1.2, transform=ax.transAxes))
    ax.add_patch(Rectangle((0, 0), 0.035, 1, fc=col, ec="none", transform=ax.transAxes))
    ax.text(0.08, 0.92, r["area"].upper(), fontsize=8.5, color=GREY, weight="bold", va="center")
    ax.text(0.08, 0.81, r["kpi"], fontsize=13, color=NAVY, weight="bold", va="center")
    ax.text(0.08, 0.58, f"{r['value_%']:.1f}%", fontsize=34, color=col, weight="bold", va="center")
    ax.text(0.08, 0.38, f"Target {r['target_txt']}   |   {abs(r['gap_pts']):.1f} pts {'to go' if r['gap_pts'] > 0 else 'better'}", fontsize=9.5, color="#111827", va="center")
    ax.add_patch(Rectangle((0.08, 0.25), 0.85, 0.045, fc="#e5e7eb", ec="none"))
    ax.add_patch(Rectangle((0.08, 0.25), 0.85 * r["value_%"] / 100, 0.045, fc=col, ec="none"))
    ax.plot([0.08 + 0.85 * r["target"] / 100] * 2, [0.21, 0.335], color="black", lw=2)
    ax.text(0.08, 0.11, textwrap.fill(sub_text[k], 46), fontsize=7.6, color=GREY, va="center", linespacing=1.35)

def style(ax):
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    ax.tick_params(labelsize=8); ax.set_facecolor("white")

def bars(ax, labels, vals, k, ylab="%", extra=None):
    r = K.loc[k]; good = (lambda v: v <= r["target"]) if r["direction"] == "lower" else (lambda v: v >= r["target"])
    worst = max(vals) if r["direction"] == "lower" else min(vals)
    cols = [GREEN if good(v) else (RED if v == worst else NAVY) for v in vals]
    ax.bar(labels, vals, color=cols, width=0.62)
    for x_, v in zip(range(len(vals)), vals): ax.text(x_, v + max(vals) * 0.02, f"{v:.0f}%", ha="center", fontsize=8.5, weight="bold")
    ax.axhline(r["target"], color="black", ls="--", lw=1.1)
    ax.text(0.99, 0.985, f"- - target {r['target_txt']}", transform=ax.transAxes, ha="right", va="top", fontsize=7.5)
    ax.set_ylim(0, max(max(vals), r["target"]) * 1.3); style(ax)

def hbars(ax, labels, vals, k):
    r = K.loc[k]; good = (lambda v: v <= r["target"]) if r["direction"] == "lower" else (lambda v: v >= r["target"])
    worst = max(vals) if r["direction"] == "lower" else min(vals)
    cols = [GREEN if good(v) else (RED if v == worst else NAVY) for v in vals]
    ax.barh(labels, vals, color=cols, height=0.66)
    for y_, v in enumerate(vals): ax.text(v + 1, y_, f"{v:.0f}%", va="center", fontsize=8.5, weight="bold")
    ax.axvline(r["target"], color="black", ls="--", lw=1.1)
    ax.set_xlabel(f"- - target {r['target_txt']}", fontsize=7.5, loc="right")
    ax.set_xlim(0, 100); ax.tick_params(axis="y", labelsize=8); style(ax)

fig = plt.figure(figsize=(16, 9), facecolor=BG)
fig.text(0.02, 0.955, "FinSight Lanka | Customer Health Dashboard", fontsize=22, weight="bold", color=NAVY)
fig.text(0.02, 0.922, f"Five monthly KPIs  |  snapshot {AS_OF:%d %b %Y}  |  {len(df)} customers after data cleaning", fontsize=10.5, color=GREY)
for i, (lab, c_) in enumerate([("On / better than target", GREEN), ("Within 20% of target", AMBER), ("More than 20% off target", RED)]):
    fig.patches.append(Rectangle((0.665 + i * 0.112, 0.935), 0.009, 0.016, transform=fig.transFigure, fc=c_, ec="none"))
    fig.text(0.677 + i * 0.112, 0.943, lab, fontsize=8, va="center", color=GREY)

xs = [0.02 + i * 0.1965 for i in range(5)]; W = 0.186
for i, k in enumerate(["K1", "K2", "K3", "K4", "K5"]):
    tile(fig.add_axes([xs[i], 0.665, W, 0.235]), k)

ax = fig.add_axes([xs[0] + 0.014, 0.335, W - 0.022, 0.255]); bars(ax, b1.index.tolist(), b1["rate_%"].round(1).tolist(), "K1")
ax = fig.add_axes([xs[1] + 0.014, 0.335, W - 0.022, 0.255])
left = np.zeros(len(b2)); pal = {"Verified": GREEN, "Expired": RED, "Pending": AMBER, "Unknown": "#9ca3af"}
for st in ["Verified", "Expired", "Pending", "Unknown"]:
    ax.barh(b2.index, b2[st], left=left, color=pal[st], label=st, height=0.62)
    for y_, (l_, v_) in enumerate(zip(left, b2[st])):
        if v_ > 8: ax.text(l_ + v_ / 2, y_, f"{v_:.0f}", ha="center", va="center", fontsize=8, color="white", weight="bold")
    left += b2[st].values
ax.set_xlim(0, 100); ax.legend(ncol=4, fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False); style(ax)
ax = fig.add_axes([xs[2] + 0.052, 0.335, W - 0.06, 0.255]); hbars(ax, b3.index.tolist(), b3["rate_%"].round(1).tolist(), "K3")
ax = fig.add_axes([xs[3] + 0.014, 0.335, W - 0.022, 0.255]); bars(ax, b4.index.tolist(), b4["rate_%"].round(1).tolist(), "K4")
for x_, (_, rr) in enumerate(b4.iterrows()): ax.text(x_, 2.5, f"LKR {rr['no_cover_savings_LKRm']:.0f}m\nuncovered", ha="center", fontsize=6.6, color="white", weight="bold", linespacing=1.2)
ax = fig.add_axes([xs[4] + 0.014, 0.335, W - 0.022, 0.255]); bars(ax, b5.index.tolist(), b5["rate_%"].round(1).tolist(), "K5")
for i, k in enumerate(["K1", "K2", "K3", "K4", "K5"]):
    fig.text(xs[i] + 0.014, 0.605, chart_title[k], fontsize=9.5, weight="bold", color="#111827")

# plain-language takeaways, generated from the numbers
worst1 = b1["rate_%"].idxmax(); best1 = b1["rate_%"].idxmin(); worstK5 = b5["rate_%"].idxmax(); worst4 = b4["rate_%"].idxmin()
acts = [("1  Loans: act on delinquency", f"Only {100-K.loc['K1','value_%']:.0f}% of loans are fully on time. {worst1}-sourced loans are the weakest ({b1.loc[worst1,'rate_%']:.0f}% delinquent) and {best1} the strongest ({b1.loc[best1,'rate_%']:.0f}%). Tighten checks on the weaker channels."),
        ("2  Compliance: fix KYC first", f"{100-K.loc['K2','value_%']:.0f}% of customers lack verified KYC, including {hi_risk_unverified} flagged PEP/suspicious customers. Start with those {hi_risk_unverified}; then clear {need_kyc} expired/pending files."),
        ("3  Growth: deepen relationships", f"LKR {nocross['Savings_Balance'].sum()/1e6:,.0f}m sits with customers holding no FD or insurance; {worst4} coverage is the lowest ({b4.loc[worst4,'rate_%']:.0f}%). {worstK5} customers show the most thin-cushion saving ({b5.loc[worstK5,'rate_%']:.0f}%).")]
for i, (h, t_) in enumerate(acts):
    x0 = 0.02 + i * 0.3275
    fig.patches.append(Rectangle((x0, 0.075), 0.318, 0.185, transform=fig.transFigure, fc="white", ec="#d0d7de", lw=1.2))
    fig.text(x0 + 0.012, 0.235, h, fontsize=11, weight="bold", color=NAVY, va="center")
    fig.text(x0 + 0.012, 0.15, textwrap.fill(t_, 56), fontsize=10.3, color="#111827", va="center", linespacing=1.45)
fig.text(0.02, 0.035, f"Source: FinSight Lanka customer file (508 raw rows -> {len(df)} customers after de-duplication and cleaning).  Loan KPI uses {int(K.loc['K1','denominator'])} loans with complete data ({n_def} defaults), so channel and segment differences are indicative, not conclusive.", fontsize=7.8, color=GREY)
fig.text(0.02, 0.014, "Bars: green = meets target, red = worst cohort, blue = other.  Dashed line = recommended target.  KPIs are calculated by compute_kpis() and can be re-run monthly.", fontsize=7.8, color=GREY)
page1 = fig
plt.show()

In [ ]:
# ---- page 2: KPI dictionary ------------------------------------------------------------------------------------------
fig2 = plt.figure(figsize=(16, 9), facecolor="white")
fig2.text(0.02, 0.955, "KPI dictionary: what each number means and why the target was chosen", fontsize=19, weight="bold", color=NAVY)
heads = [("KPI", 0.02), ("Definition & formula", 0.185), ("Now vs target", 0.445), ("Why it matters / decision it informs", 0.525), ("Why this target", 0.75)]
for h, x in heads: fig2.text(x, 0.915, h, fontsize=10.5, weight="bold", color="white", bbox=dict(boxstyle="square,pad=0.35", fc=NAVY, ec="none"))
rowh = 0.168
for i, k in enumerate(["K1", "K2", "K3", "K4", "K5"]):
    r = K.loc[k]; y = 0.865 - i * rowh
    fig2.patches.append(Rectangle((0.015, y - rowh + 0.012), 0.97, rowh - 0.012, transform=fig2.transFigure, fc="#f9fafb" if i % 2 == 0 else "white", ec="#e5e7eb"))
    fig2.patches.append(Rectangle((0.015, y - rowh + 0.012), 0.006, rowh - 0.012, transform=fig2.transFigure, fc=RAGC[r["RAG"]], ec="none"))
    fig2.text(0.027, y - 0.02, textwrap.fill(f"{k}  {r['kpi']}", 20), fontsize=10.5, weight="bold", color=NAVY, va="top", linespacing=1.2)
    fig2.text(0.027, y - 0.08, f"Area: {r['area']}\n" + textwrap.fill(f"Owner: {r['owner']}", 30), fontsize=8, color=GREY, va="top", linespacing=1.4)
    fig2.text(0.185, y - 0.02, textwrap.fill(r["definition"], 46) + "\n\nFormula: " + textwrap.fill(r["formula"], 46).replace("\n", "\n         "), fontsize=7.6, va="top", linespacing=1.3)
    fig2.text(0.445, y - 0.02, f"{r['value_%']:.1f}%", fontsize=19, weight="bold", color=RAGC[r["RAG"]], va="top")
    fig2.text(0.445, y - 0.075, f"Target {r['target_txt']}\n95% CI {r['ci_low_%']:.0f}-{r['ci_high_%']:.0f}%", fontsize=8.2, va="top", linespacing=1.4)
    fig2.text(0.525, y - 0.02, textwrap.fill(r["why"], 47) + "\n\nDecision: " + textwrap.fill(r["decision"], 47), fontsize=7.6, va="top", linespacing=1.3)
    fig2.text(0.75, y - 0.02, textwrap.fill(r["target_logic"], 58), fontsize=7.6, va="top", linespacing=1.3)
fig2.text(0.02, 0.012, "RAG rule: Green = target met; Amber = within 20% of target; Red = >20% away. 'Delayed' is broader than a formal non-performing loan (NPL); compare K1 with CBSL's NPL ratio cautiously.", fontsize=7.8, color=GREY)
page2 = fig2
plt.show()

with PdfPages(OUT / "FinSightAssessment_Dashboard.pdf") as pdf:
    pdf.savefig(page1, facecolor=page1.get_facecolor()); pdf.savefig(page2, facecolor="white")
page1.savefig(OUT / "dashboard_page1.png", dpi=130, facecolor=page1.get_facecolor()); page2.savefig(OUT / "dashboard_page2.png", dpi=130)
print("Saved dashboard PDF + PNGs to", OUT.resolve())

### Dashboard design note

The dashboard in this notebook is a **working structure and analytical prototype** to demonstrate the selected KPIs, calculations, and supporting views. My final Tableau workbook may use a different visual design, layout, colours, filters, and interactions to suit the audience and improve usability. Any design changes will preserve the KPI definitions, calculations, data-quality caveats, and conclusions documented here.

---
## Q4.6 Exports for Tableau 
Everything needed to rebuild the dashboard in a BI tool: the clean customer table (one row per customer, flags included) plus the KPI and breakdown tables. Step-by-step Tableau and Power BI build guides are supplied separately.

In [ ]:
clean_export = df.drop(columns=["Full_Name", "NIC_Number"], errors="ignore")
clean_export.to_csv(OUT / "FinSight_Dashboard_Data.csv", index=False)
kt_out = kt[["id", "area", "kpi", "numerator", "denominator", "value_%", "ci_low_%", "ci_high_%", "target", "direction", "gap_pts", "RAG",
             "definition", "formula", "why", "decision", "target_logic", "owner", "trigger"]]
with pd.ExcelWriter(OUT / "Q4_KPI_Workings.xlsx", engine="openpyxl") as xw:
    kt_out.round(2).to_excel(xw, sheet_name="KPI_Table", index=False)
    b1.round(2).to_excel(xw, sheet_name="K1_by_channel"); b2.round(2).to_excel(xw, sheet_name="K2_KYC_mix")
    b3.round(2).to_excel(xw, sheet_name="K3_by_province"); b4.round(2).to_excel(xw, sheet_name="K4_by_segment"); b5.round(2).to_excel(xw, sheet_name="K5_by_channel")
    sens.to_excel(xw, sheet_name="Sensitivity", index=False); pd.DataFrame(cleaning_log).to_excel(xw, sheet_name="Cleaning_Log", index=False)
    for ws in xw.book.worksheets:
        ws.freeze_panes = "A2"
        for colx in ws.columns:
            w = max(len(str(c.value)) if c.value is not None else 0 for c in colx[:30])
            ws.column_dimensions[colx[0].column_letter].width = min(max(w + 2, 10), 60)
print(sorted(p.name for p in OUT.iterdir()))

---
## Q4.7 Monthly tracking template
Each month: load the new extract, apply the same cleaning, call `compute_kpis()`, append. 

In [ ]:
history = kpi_now[["id", "kpi", "value_%", "ci_low_%", "ci_high_%"]].copy()
history.insert(0, "month", AS_OF.strftime("%Y-%m"))
history.to_csv(OUT / "kpi_history.csv", index=False)
display(history.round(1))
print("Next month: kpi_now = compute_kpis(df_next); history = pd.concat([history, kpi_now.assign(month='YYYY-MM')]) ->  trend lines + RAG triggers (see 'trigger' column in KPI table).")

---
# Q5. Board summary and recommendation

**Headline:** *Our customers are saving, but the loan book is the weak point, nearly a third of customers are not fully identity-verified, and the biggest growth opportunity is already inside our existing Premium customers.*

## Four findings 

**1. Only 6 in 10 loans are fully on time.**
Of 218 loans (LKR 168M outstanding), 133 are on time, 63 are late and 22 have defaulted. Late or defaulted loans account for LKR 68M, about 41% of the money lent. Loans brought in through Referrals are the weakest (47% late or defaulted); Agent-sourced loans are the strongest (32%). *These are small numbers, so the channel differences are a warning signal, not proof.*

**2. A small group holds most of the money, and many have not been offered more.**
Premium customers are 18% of customers but hold 64% of all savings, yet only 36% of them have a fixed deposit or insurance, the lowest of any segment. In total, 251 customers hold LKR 222M (59% of all savings) with neither product. Nobody is draining their savings, but 1 in 6 customers withdraws most of what they deposit each month, so their cushion is thin (28% of Branch customers).

**3. Nearly 1 in 3 customers is not fully identity-verified (KYC).**
149 of 500 customers (30%) are not verified, and they hold LKR 121M (32%) of savings. Ten of them are flagged as politically exposed or suspicious. This is a regulatory and reputational risk, so it comes first.

**4. The app and the acquisition channel are not the growth levers people may assume.**
App users hold no more than other customers (typical balance about LKR 449k vs 445k). No single channel wins on both balances and loan quality: Agent is the safest for lending but has the smallest balances, while Digital has the best typical balance but the weakest ID checks and account activity. Budget should not move on one metric alone.

*Data note:* the raw file had 508 rows for 500 customers, with placeholder values and contradictions. We cleaned it (Section 0), and recommend fixing the source data too.

## Two strategic recommendations

**Recommendation 1: Stop good loans going bad (90-day early-collections and lending-guardrail programme).**
- **Act on the 63 late loans (LKR 51M, about LKR 0.8M each)** with early repayment plans *before* they roll into default. They have not defaulted yet, so this is the cheapest place to prevent losses.
- **Tighten new lending as a precaution:** require verified ID and an affordability check, with extra care on Referral-sourced loans, Vehicle loans (8 of 51 defaulted) and Starter customers (6 of 36).
- **Target:** cut the late/default share from 39% to 32% in 12 months (the level Agent already achieves) and defaults to 6% or less. Seven points is about 15 fewer troubled loans, or roughly LKR 12M of balances back on track.
- **Owner and measure:** Chief Risk Officer; tracked monthly as K1.
- **Caveat:** only 22 loans have defaulted, so the specific watch-list items are precautions to be confirmed with repayment history (data source 1 below), and the 32% target should be validated by a pilot (see Q6, criticism 1).

**Recommendation 2: Fix identity checks first, then run a targeted fixed-deposit and insurance campaign.**
- **Step 1, within 30 days:** verify the 10 flagged customers, then clear the 149 unverified files, starting with customers who are ready to buy: 56 customers (LKR 40M) are blocked only by an expired or pending KYC.
- **Step 2, within 90 days:** relationship managers contact the 95 customers who are active, verified and not over-borrowed (LKR 91M), leading with Premium.
- **Target:** KYC-verified from 70% to 98% (the regulatory floor is 100%) and cross-sell cover from 48.7% to 55% (about 31 more customers, roughly one in three of the 95).
- **Why this one:** it is low-regret. The compliance step is required anyway, and it unlocks the growth step.
- **Owners and measures:** Head of Compliance (K2) and Head of Retail Products (K4).

**Deliberately not recommended:** shifting the whole acquisition budget to one channel, or investing in the app to grow balances. The data does not support either.

## Two additional data sources
1. **Monthly loan repayment and account history** (payment dates, days past due, monthly balances, closure dates). A single snapshot cannot show whether "late" means 5 days or 90, whether delinquency is rising or falling, how many late loans roll into default, or who is drifting toward draining or leaving. History gives a true non-performing-loan figure comparable with the sector, an early-warning model, and real trends for the monthly KPIs.
2. **Customer income and credit-bureau data (for example CRIB).** The debt-to-savings measure ignores income, other lenders and fixed deposits, and it did not predict defaults here (a limit of the measure, not proof that affordability does not matter). Income and bureau history give a real affordability picture, better lending decisions, and less reliance on a small default sample.

In [ ]:
# ---- Q5 evidence: every figure quoted above, re-computed from the cleaned data --------------------------------------
Lc = df[df["Loan_Data_Complete"]]; stt = Lc["Loan_Repayment_Status"]; book = Lc["Outstanding_Loan_Balance"].sum()
late = stt.isin(["Delayed", "Defaulted"])
tot_sav = df["Savings_Balance"].sum()
prem = df[df["Customer_Segment"] == "Premium"]
vs = df[df["Savings_Balance"] > 0]
pool = vs[(vs["Has_Fixed_Deposit"] == "No") & (vs["Has_Insurance"] == "No")]
nv = df[df["KYC_Status"] != "Verified"]
flagged_unv = ((df["PEP_Flag"] == "Yes") | (df["Suspicious_Activity_Flag"] == "Yes")) & (df["KYC_Status"] != "Verified")
wr_ = df["Monthly_Withdrawal_Avg"] / df["Monthly_Deposit_Avg"]; okw = wr_.notna()
app_med = df.loc[(df["Mobile_App_User"] == "Yes") & df["Savings_Balance"].notna(), "Savings_Balance"].median()
non_med = df.loc[(df["Mobile_App_User"] == "No") & df["Savings_Balance"].notna(), "Savings_Balance"].median()

act_ = pool["Account_Status"] == "Active"; no_flag = (pool["Suspicious_Activity_Flag"] != "Yes") & (pool["PEP_Flag"] != "Yes"); ver_ = pool["KYC_Status"] == "Verified"
_LD = df[df["Loan_Data_Complete"] & (df["Savings_Balance"] > 0)]
stretched_ids = set(_LD.loc[_LD["Outstanding_Loan_Balance"] / _LD["Savings_Balance"] > 1, "Customer_ID"])
tierA = pool[act_ & ver_ & no_flag]; tierA_safe = tierA[~tierA["Customer_ID"].isin(stretched_ids)]
kyc_only = pool[act_ & ~ver_ & no_flag]
covered = int(((vs["Has_Fixed_Deposit"] == "Yes") | (vs["Has_Insurance"] == "Yes")).sum())
need_cov = int(np.ceil(0.55 * len(vs))) - covered
avg_late = Lc.loc[late, "Outstanding_Loan_Balance"].mean()
ch_del = Lc.groupby("Acquisition_Channel").apply(lambda g: g["Loan_Repayment_Status"].isin(["Delayed", "Defaulted"]).mean() * 100, include_groups=False)
fr_ch = df[okw].assign(fr=(wr_[okw] > 0.8)).groupby("Acquisition_Channel")["fr"].mean() * 100

rows = []
def chk(label, value, quoted, tol=0.5):
    rows.append({"Figure quoted in Q5": label, "Computed": round(float(value), 2), "Quoted": quoted, "Matches": "yes" if abs(float(value) - quoted) <= tol else "CHECK"})
chk("Loans with complete data", len(Lc), 218, 0)
chk("Outstanding loan book (LKR m)", book / 1e6, 168, 0.6)
chk("On-time loans", (stt == "On-Time").sum(), 133, 0); chk("Delayed (late) loans", (stt == "Delayed").sum(), 63, 0); chk("Defaulted loans", (stt == "Defaulted").sum(), 22, 0)
chk("Late + defaulted exposure (LKR m)", Lc.loc[late, "Outstanding_Loan_Balance"].sum() / 1e6, 68, 0.6)
chk("... as % of money lent", Lc.loc[late, "Outstanding_Loan_Balance"].sum() / book * 100, 41, 0.6)
chk("Delayed-loan exposure (LKR m)", Lc.loc[stt == "Delayed", "Outstanding_Loan_Balance"].sum() / 1e6, 51, 0.5)
chk("Average late loan balance (LKR m)", avg_late / 1e6, 0.8, 0.05)
chk("Referral late/default %", ch_del["Referral"], 47, 0.6); chk("Agent late/default %", ch_del["Agent"], 32, 0.6)
chk("Late/default rate overall %", late.mean() * 100, 39, 0.6)
chk("Premium % of customers", len(prem) / len(df) * 100, 18, 0.6); chk("Premium % of savings", prem["Savings_Balance"].sum() / tot_sav * 100, 64, 0.6)
chk("Premium cover (FD or insurance) %", ((prem["Has_Fixed_Deposit"] == "Yes") | (prem["Has_Insurance"] == "Yes"))[prem["Savings_Balance"] > 0].mean() * 100, 36, 0.6)
chk("No-FD-no-insurance customers", len(pool), 251, 0); chk("... savings (LKR m)", pool["Savings_Balance"].sum() / 1e6, 222, 0.6)
chk("... % of all savings", pool["Savings_Balance"].sum() / tot_sav * 100, 59, 0.6)
chk("Fragile savers % (withdraw >80% of deposits)", (wr_[okw] > 0.8).mean() * 100, 16, 0.6); chk("Branch fragile %", fr_ch["Branch"], 28, 0.6)
chk("Customers not KYC-verified", len(nv), 149, 0); chk("... % of customers", len(nv) / len(df) * 100, 30, 0.6)
chk("... savings (LKR m)", nv["Savings_Balance"].sum() / 1e6, 121, 0.6); chk("... % of savings", nv["Savings_Balance"].sum() / tot_sav * 100, 32, 0.6)
chk("PEP/suspicious customers not verified", flagged_unv.sum(), 10, 0)
chk("Median savings: app users (LKR k)", app_med / 1e3, 449, 0.6); chk("Median savings: non-users (LKR k)", non_med / 1e3, 445, 0.6)
chk("Ready-now customers (Tier A, not over-borrowed)", len(tierA_safe), 95, 0); chk("... savings (LKR m)", tierA_safe["Savings_Balance"].sum() / 1e6, 91, 0.6)
chk("Blocked only by KYC (active, no flags)", len(kyc_only), 56, 0); chk("... savings (LKR m)", kyc_only["Savings_Balance"].sum() / 1e6, 40, 0.6)
chk("Extra covered customers needed for 55% cover", need_cov, 31, 0)
vt = Lc[Lc["Loan_Type"] == "Vehicle"]; chk("Vehicle defaults", (vt["Loan_Repayment_Status"] == "Defaulted").sum(), 8, 0); chk("Vehicle loans", len(vt), 51, 0)
st_ = Lc[Lc["Customer_Segment"] == "Starter"]; chk("Starter defaults", (st_["Loan_Repayment_Status"] == "Defaulted").sum(), 6, 0); chk("Starter loans", len(st_), 36, 0)
chk("7-point delinquency cut = loans", 0.07 * len(Lc), 15, 0.5); chk("... balances back on track (LKR m)", 0.07 * len(Lc) * avg_late / 1e6, 12, 0.6)
q5_evidence = pd.DataFrame(rows); display(q5_evidence)
assert (q5_evidence["Matches"] == "yes").all(), "A figure quoted in Q5 does not match the data"
print("All", len(q5_evidence), "figures quoted in the board summary are reproduced by the data.")
q5_evidence.to_csv(OUT_ROOT / "q5_evidence.csv", index=False)

---
# Q6. Critique of my own work
I stress-tested my own choices before writing this. Some held up (duplicate resolution; the dependence of Q1B on a few large balances). Others did not: the targets, the loan watch-list and the activity KPI. The tests behind every number below are in the evidence cells that follow.

| # | Criticism | Severity |
|---|---|---|
| 1 | Targets rest on the best of four small cohorts (winner's curse) | **High** |
| 2 | One-month snapshot that looks bounded, with no closed accounts | **High** |
| 3 | The loan watch-list may be noise (about 70 groups tested) | Medium |
| 4 | "Active" status does not behave like engagement | Medium |
| 5 | Cross-sell opportunity is existing balances, and concentrated | Medium |
| 6 | Debt-to-savings is a weak risk lens | Medium |
| 7 | The app flag is inconsistent, which may hide a real effect | Medium |
| 8 | Cleaning judgement calls (tested: little impact) | Low |

### 1. Targets rest on the best of four small cohorts (High)
- **Limitation:** I justified the 32% delinquency target because "Agent already achieves it". But Agent's figure rests on 56 loans, with a 95% interval of roughly 21-45%. If all four channels truly had the same 39% rate, there is about a 45% chance that the best-looking one would show 32% or lower by luck alone.
- **Why it matters:** the same selection effect applies to my other "best cohort" targets (K3, K5), to the claim that Agent is the safest channel (3 defaults in 56 loans), and to the "15 fewer troubled loans" estimate in Recommendation 1.
- **Remedy:** run a six-month pilot before committing to the target; use pooled estimates that shrink small cohorts toward the portfolio average; wait for more loans; anchor the target on the CBSL sector benchmark.

### 2. One-month snapshot that looks bounded, with no closed accounts (High)
- **Limitation:** every customer withdraws between 30% and 90% of what they deposit, and every net flow is positive (the smallest is LKR 31). Real customers do not behave this uniformly, so the data looks constructed or capped. Closed accounts are also absent, so customers who drained their savings and left cannot appear (survivorship).
- **Why it matters:** "nobody is draining" is probably a data artefact, not good news. The fragile-saver KPI is my own construct, is threshold-dependent (about 33% at a 70% cut-off, 16% at 80%, 0% at 90%) and was never validated against actual later drains. Monthly averages also hide within-month swings.
- **Remedy:** monthly balance history plus closed-account records, then back-test K5 on whether flagged customers actually drain later.

### 3. The loan watch-list may be noise (Medium)
- **Limitation:** Q2C examined 69 groups and cells. The strongest result, Starter × Vehicle (4 of 13), has p = 0.035, which becomes about 0.42 after adjusting for 12 comparisons. The omnibus tests found no reliable pattern.
- **Why it matters:** I labelled these a watch-list, but Recommendation 1 still builds precautionary guardrails on them.
- **Remedy:** pre-specify a few hypotheses, adjust for multiple tests, and validate on next quarter's loans before tightening lending.

### 4. "Active" status does not behave like engagement (Medium)
- **Limitation:** 115 of the 123 non-active accounts still transact (a median of 9 transactions vs 10 for active accounts) and 121 still deposit. 29 of the 43 suspended accounts have verified KYC.
- **Why it matters:** K3, and the claim that 29% of savings sit in non-active accounts, may measure an administrative code rather than customer behaviour.
- **Remedy:** define activity from dated transactions (for example no transaction in 90 days) and clarify the status codes with the data owner.

### 5. Cross-sell opportunity is existing balances, and concentrated (Medium)
- **Limitation:** the LKR 222M is money already held; moving it into a fixed deposit is mostly a transfer, not new money. It is also concentrated: the top 25 customers hold 45% of it, and 109 of the 251 also hold loans.
- **Why it matters:** a board could read it as revenue. It is the size of a pool, not profit.
- **Remedy:** obtain product margins and past campaign response rates, estimate net new funds, and prioritise a short named list.

### 6. Debt-to-savings is a weak risk lens (Medium)
- **Limitation:** it ignores income, other lenders and fixed deposits (FD amounts exceed savings for 108 of the 140 FD holders with a valid balance), so the count swings from 127 to 101 depending on whether FDs are counted. It also did not predict defaults (p = 0.64), yet I still used it to screen cross-sell targets.
- **Remedy:** add income and credit-bureau data and build a proper affordability measure.

### 7. The app flag is inconsistent, which may hide a real effect (Medium)
- **Limitation:** 49 non-users list the app as their preferred channel and 86 have push notifications on. Misclassifying customers pushes any real difference toward zero (attenuation), so "no relationship" may be partly an artefact. My alternative definitions use the same noisy fields.
- **Remedy:** use real login and usage logs and re-run the comparison.

### 8. Cleaning judgement calls (Low)
- **Limitation:** several judgement calls: five balances set to missing (not corrected), segment inferred with ~85% accuracy, province rebuilt from district, and unresolved contradictions (122 KYC dates before the account opened).
- **Why Low:** the duplicate and large-balance choices were tested and barely move anything (below).
- **Remedy:** confirm the placeholder values and status codes with the data owner and request a data dictionary.

### How confident to be in each finding
| Finding | Confidence |
|---|---|
| 30% of customers not KYC-verified | **High** (direct count) |
| Premium concentration; LKR 222M cross-sell pool | **High** for counts, **Medium** for value |
| 39% of loans delayed or defaulted | **Medium-High** (95% interval about 33-46%) |
| Channel and segment loan patterns; app effect | **Low-Medium** |

In [ ]:
# ---- Q6 evidence: the tests behind each criticism ------------------------------------------------------------------
ev = []
def add(n, what, value): ev.append({"#": n, "Test": what, "Result": value}); print(f"[{n}] {what}: {value}")

# 1. winner's curse: if all four channels truly had the same delinquency rate, how often would the best look this good?
P_TRUE, N_PER = 0.39, 55
sims = np.random.default_rng(1).binomial(N_PER, P_TRUE, size=(100_000, 4)) / N_PER
agent = Lc[Lc["Acquisition_Channel"] == "Agent"]
k_a, n_a = int(agent["Loan_Repayment_Status"].isin(["Delayed", "Defaulted"]).sum()), len(agent)
lo_, hi_ = stats.binomtest(k_a, n_a).proportion_ci(method="wilson")
add(1, f"Agent delinquency {k_a}/{n_a}: 95% CI", f"{lo_:.1%} to {hi_:.1%}")
add(1, "P(best of 4 equal channels looks <= 32.1% by luck alone)", f"{(sims.min(1) <= 0.321).mean():.1%}")
d_a = int((agent["Loan_Repayment_Status"] == "Defaulted").sum()); lo2, hi2 = stats.binomtest(d_a, n_a).proportion_ci(method="wilson")
add(1, f"Agent default {d_a}/{n_a}: 95% CI", f"{lo2:.1%} to {hi2:.1%}")

# 2. bounded / snapshot data and K5 threshold sensitivity
add(2, "Withdrawal / deposit ratio range across customers", f"{wr_.min():.2f} to {wr_.max():.2f}")
add(2, "Share of customers with positive net monthly flow; smallest flow (LKR)", f"{(df['Net_Monthly_Flow'] > 0).mean():.0%}; {df['Net_Monthly_Flow'].min():.2f}")
add(2, "Fragile-saver rate at 70% / 80% / 90% thresholds", " / ".join(f"{(wr_[okw] > t).mean():.1%}" for t in (0.7, 0.8, 0.9)))

# 3. multiple testing
p_sv = stats.binomtest(4, 13, 22 / 218, alternative="greater").pvalue
add(3, "Groups/cells examined in Q2C", len(allt))
add(3, "Starter x Vehicle p-value; Bonferroni-adjusted for 12 cells", f"{p_sv:.3f}; {min(p_sv * 12, 1):.2f}")

# 4. status vs behaviour
na_ = df[df["Account_Status"] != "Active"]
add(4, "Non-active accounts that still transact / deposit", f"{(na_['Monthly_Transaction_Count'] > 0).sum()} / {(na_['Monthly_Deposit_Avg'] > 0).sum()} of {len(na_)}")
add(4, "Median monthly transactions: non-active vs active", f"{na_['Monthly_Transaction_Count'].median():.0f} vs {df.loc[df['Account_Status'] == 'Active', 'Monthly_Transaction_Count'].median():.0f}")
sus = df[df["Account_Status"] == "Suspended"]
add(4, "Suspended accounts with verified KYC", f"{(sus['KYC_Status'] == 'Verified').sum()} of {len(sus)}")

# 5. concentration of the cross-sell pool
ps = pool["Savings_Balance"].sort_values(ascending=False)
add(5, "Share of the pool held by the top 10 / top 25 customers", f"{ps.head(10).sum() / ps.sum():.0%} / {ps.head(25).sum() / ps.sum():.0%}")
add(5, "Pool customers who also hold a loan", f"{int(pool['Loan_Holder'].sum())} of {len(pool)}")

# 6. debt-to-savings as a risk lens
add(6, "FD_Amount exceeds Savings_Balance (FD holders with a valid balance)", f"{int((df['FD_Amount'] > df['Savings_Balance']).sum())} of {int((df['FD_Amount'].notna() & df['Savings_Balance'].notna()).sum())}")
add(6, "DSR>1 customers: savings only vs savings + FD", f"{q2_sens.loc[0, 'DSR>1 customers']} vs {q2_sens.loc[3, 'DSR>1 customers']}")
add(6, "Fisher test: DSR>1 vs default (p-value)", f"{p_dsr_def:.2f}")

# 7. app flag contradictions
add(7, "Non-users who prefer the App channel / have push notifications on", f"{int(df['Flag_App_Channel_Contradiction'].sum())} / {int(df['Flag_Push_Without_App'].sum())}")

# 8. robustness of cleaning choices
k2_res = (df["KYC_Status"] == "Verified").mean() * 100
q_ = df[~df["Customer_ID"].isin(CONFLICT_IDS)]
add(8, "KYC-verified rate: duplicates resolved vs quarantined", f"{k2_res:.1f}% ({int((df['KYC_Status']=='Verified').sum())}/{len(df)}) vs {(q_['KYC_Status']=='Verified').mean()*100:.1f}% ({int((q_['KYC_Status']=='Verified').sum())}/{len(q_)})")
bins_ = [18, 25, 35, 45, 55, 65, np.inf]; labs_ = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
def top_band(d_):
    t = d_.assign(b=pd.cut(d_["Age"], bins_, labels=labs_, right=False)).groupby("b", observed=False)["Savings_Balance"].sum()
    return t.idxmax(), t.max() / t.sum() * 100
big10 = df.nlargest(10, "Savings_Balance")["Customer_ID"]
b_all, b_ex = top_band(df), top_band(df[~df["Customer_ID"].isin(big10)])
add(8, "Top age band by total savings: all customers vs excluding the 10 largest balances", f"{b_all[0]} ({b_all[1]:.1f}%) vs {b_ex[0]} ({b_ex[1]:.1f}%)")
add(8, "KYC dates earlier than account opening", int(df["Flag_KYC_Before_Open"].sum()))
q6_evidence = pd.DataFrame(ev); q6_evidence.to_csv(OUT_ROOT / "q6_evidence.csv", index=False)

---
## End of workings: files produced

In [ ]:
for p in sorted(OUT_ROOT.rglob("*")):
    if p.is_file(): print(p.as_posix(), f"({p.stat().st_size/1024:,.0f} KB)")